# Laguna XS.2 v10 — Behavior-Aligned Writeability

## Can writable locations produce **better autonomous behavior** when both location and training dose are selected correctly?

v9 established two facts:

1. raw gradient magnitude identifies parameters that are very easy to move;
2. large teacher-forced NLL improvements do **not** necessarily improve free-running GSM8K accuracy.

v10 is an adaptive follow-up designed specifically around that failure mode. It does **not**
reuse the v9 final examples. It reconstructs every v9-used row from the pinned datasets,
excludes those IDs, and creates a fresh selection/train/calibration/final protocol.

### What v10 fixes logically

- **Fresh final test:** the v9 final set is no longer untouched after we inspected it, so v10
  uses a disjoint GSM8K/MBPP final set.
- **Behavior selects dose:** learning rate and checkpoint selection use held-out
  **GSM8K generation accuracy**, not teacher-forced NLL.
- **Control movement is symmetric:** both positive and negative MBPP NLL movement count as
  non-specific change via `abs(control_improvement)`. v9 only penalized control damage.
- **Parameter-family-matched guidance:** attention LoRA locations are chosen from attention
  gradients, expert locations from expert gradients.
- **Contrastive gradient geometry:** the new selector uses the component of the target gradient
  orthogonal to the control gradient, rather than only raw norm or target-minus-control norm.
- **Independent calibration stages:** family LR calibration and method dose calibration use
  disjoint held-out rows.
- **Three-seed fresh confirmation:** all final decisions are frozen before the fresh final set is scored.

### Final methods

1. `gradient_experts`
2. `gradient_specific_experts`
3. `contrastive_experts`
4. `standard_lora`
5. `random_layer_lora`
6. `contrastive_lora`

All methods retain the same ~K=4 expert-equivalent trainable-parameter budget. The v9 fixed-64-update
experiment already tested equal update count; v10 instead tests the **practical policy**:
select a family LR and a method-specific early-stopping dose from held-out behavior, then confirm once
on a fresh final set.

## 1 — Install dependencies once

In [ ]:
# Keep the CUDA-enabled PyTorch build supplied by the AWS image.
# Run this cell once, restart the kernel once, then Run All.
%pip -q install \
  "transformers==5.14.1" \
  "peft==0.19.1" \
  "datasets>=4.0,<5" \
  "accelerate>=1.10.0" \
  "huggingface_hub>=0.35.0" \
  safetensors pandas numpy psutil tqdm matplotlib packaging tabulate

print("Dependencies installed. Restart the kernel once if packages changed.")

## 2 — Runtime tuning for g7e.2xlarge

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["MALLOC_ARENA_MAX"] = "4"

os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "2"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "8"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:512"
)

print("Runtime configured for AWS g7e.2xlarge / v9.")

## 3 — Canonical imports

In [ ]:
import os
import gc
import json
import math
import time
import types
import shutil
import hashlib
import platform
import sys
import importlib.util
import re
import uuid
import zipfile

from pathlib import Path
from contextlib import contextmanager, ExitStack
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from fractions import Fraction

import numpy as np
import pandas as pd
import psutil

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

print("Core imports: PASS")
print("Torch:", torch.__version__)

## 4 — Dependency verification

In [ ]:
from importlib.metadata import version as package_version
from packaging.version import Version

required_modules = [
    "torch", "transformers", "peft", "datasets", "accelerate",
    "huggingface_hub", "safetensors", "numpy", "pandas", "psutil",
    "tqdm", "matplotlib", "tabulate",
]
missing = [x for x in required_modules if importlib.util.find_spec(x) is None]
if missing:
    raise ModuleNotFoundError(
        "Missing required modules: " + ", ".join(missing)
        + ". Re-run the install cell, restart, then Run All."
    )

transformers_version = package_version("transformers")
peft_version = package_version("peft")

if transformers_version != "5.14.1":
    raise RuntimeError(
        f"Requires transformers==5.14.1; found {transformers_version}."
    )
if Version(peft_version) < Version("0.19.1"):
    raise RuntimeError(
        f"Transformers v5 PEFT integration requires peft>=0.19.1; found {peft_version}."
    )
if importlib.util.find_spec("transformers.conversion_mapping") is None:
    raise ModuleNotFoundError("transformers.conversion_mapping unavailable.")

print("Dependency verification: PASS")
print("Transformers:", transformers_version)
print("PEFT:", peft_version)
print("Datasets:", package_version("datasets"))

## 5 — Hardware and storage preflight

In [ ]:
import os
import shutil
import platform
from pathlib import Path

import psutil
import torch

ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print("Logical CPUs:", os.cpu_count())
print(f"RAM total:     {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f"This notebook expects exactly one GPU; found {torch.cuda.device_count()}."
    )

props = torch.cuda.get_device_properties(0)

print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory / 2**30 < 88:
    raise RuntimeError("Need a 96GB-class GPU (~89 GiB binary or larger).")

if (os.cpu_count() or 0) < 8:
    print("WARNING: fewer than 8 logical CPUs detected.")

if ram.total / 2**30 < 58:
    print("WARNING: less than a 64-GiB-class host detected.")

roots = [
    Path("/home/ec2-user/workspace"),
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

choices = []
seen_devices = set()

for p in roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen_devices:
            continue
        seen_devices.add(dev)
        usage = shutil.disk_usage(p)
        choices.append((usage.free, p, usage))
    except OSError:
        pass

if not choices:
    raise RuntimeError("No writable filesystem found.")

_, WORK_ROOT, disk = max(choices, key=lambda x: x[0])

print("\n=== Storage ===")
print("Work root:", WORK_ROOT)
print(f"Disk free: {disk.free/2**30:.2f} GiB")
print("Hardware preflight: PASS")

## 6 — v10 experiment configuration

In [ ]:
TARGET_DATASET_ID = "openai/gsm8k"
TARGET_DATASET_CONFIG = "main"
TARGET_DATASET_REVISION = "e53f048856ff4f594e959d75785d2c2d37b678ee"

CONTROL_DATASET_ID = "google-research-datasets/mbpp"
CONTROL_DATASET_CONFIG = "full"
CONTROL_DATASET_REVISION = "4bb6404fdc6cacfda99d4ac4205087b89d32030c"

# v9 exclusion protocol. These values reconstruct the exact rows already seen in v9.
V9_DATA_SPLIT_SEED = 93017
V9_SELECTION_TARGET_N = 32
V9_TRAIN_TARGET_N = 256
V9_VALIDATION_TARGET_N = 64
V9_FINAL_TARGET_N = 128
V9_SELECTION_CONTROL_N = 32
V9_VALIDATION_CONTROL_N = 64
V9_FINAL_CONTROL_N = 128

# Prefer the exact v9 snapshot from the previous run if it still exists on disk.
# If not, deterministically reconstruct the same IDs from the pinned v9 protocol.
V9_SNAPSHOT_CANDIDATE = (
    WORK_ROOT
    / "laguna_xs2_v9_matched_peft_gsm8k"
    / "benchmark_snapshot.csv"
)
EXPECTED_COMPLETED_V9_SNAPSHOT_SHA256 = (
    "8cedb762f6d18ef0dc2f3a828bae64d68da03643238aa2659d93a65a289e361b"
)

# Fresh v10 deterministic partition, after excluding all v9-used IDs.
DATA_SPLIT_SEED = 104729

SELECTION_TARGET_N = 48
SELECTION_CONTROL_N = 48

TRAIN_TARGET_N = 256

# Disjoint calibration sets:
#  - LR calibration chooses one LR per parameter family at a fixed small dose.
#  - Dose calibration chooses method-specific early stopping using the frozen family LR.
LR_CAL_TARGET_N = 48
LR_CAL_CONTROL_N = 48

DOSE_CAL_TARGET_N = 64
DOSE_CAL_CONTROL_N = 64

# Fresh, v9-disjoint confirmatory final set.
FINAL_TARGET_N = 192
FINAL_CONTROL_N = 192
GENERATION_EVAL_N = FINAL_TARGET_N

# Selection probes.
CONTROL_PENALTY = 0.75  # legacy v9 gradient-specific expert score only
GLOBAL_GRAD_PROBE_CASES = 16
GLOBAL_GRAD_CHUNK = 8
PRECISE_GRAD_CASES = 16
ATTENTION_GRAD_PROBE_CASES = 16
SELECTOR_TOP_N = 16

EXPERT_BUDGET_K = 4
GUIDED_LORA_N_LAYERS = 8
RANDOM_LORA_N_LAYERS = 8
RANDOM_LAYER_SEED = 104731
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
MAX_LORA_BUDGET_REL_ERROR = 0.05

# Family LR calibration uses autonomous behavior at a fixed small dose.
EXPERT_LR_GRID = [2.5e-6, 5e-6, 1e-5]
LORA_LR_GRID = [1e-5, 2.5e-5, 5e-5, 1e-4]
LR_PROBE_UPDATES = 8
LR_CALIBRATION_SEED = 101

# Method-specific dose calibration, including no-op/early-stop step 0.
DOSE_CHECKPOINTS = [0, 1, 2, 4, 8, 16, 32, 64]
DOSE_CALIBRATION_SEED = 131

# New final training-order seeds; independent of v9.
FINAL_ORDER_SEEDS = [17, 43, 71]

TRAIN_EPOCHS = 2
TRAIN_MAX_UPDATES = max(DOSE_CHECKPOINTS)
TRAIN_GRAD_ACCUM = 8

OPTIMIZER_BETAS = (0.9, 0.95)
OPTIMIZER_WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0
LORA_ALPHA_MULTIPLIER = 1.0

EVAL_BATCH_SIZE = 8
GENERATION_BATCH_SIZE = 4
GENERATION_MAX_NEW_TOKENS = 256

DELTA_MERGE_WARN_NLL = 0.05

PRIMARY_METHODS = [
    "gradient_experts",
    "gradient_specific_experts",
    "contrastive_experts",
    "standard_lora",
    "random_layer_lora",
    "contrastive_lora",
]

PROTOCOL_VERSION = "v10.0-behavior-aligned-writeability"
RESULTS_ROOT = WORK_ROOT / "laguna_xs2_v10_behavior_aligned_writeability"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = RESULTS_ROOT
CURRENT_CAPABILITY = "gsm8k"

print("Results root:", RESULTS_ROOT)
print("Protocol:", PROTOCOL_VERSION)

## 7 — Crash-safe checkpoint helpers

In [ ]:
def atomic_to_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=index)
    with open(tmp, "rb") as f:
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def read_checkpoint_csv(path, required_columns=None, dedupe_keys=None):
    path = Path(path)
    required_columns = list(required_columns or [])
    if not path.exists():
        return pd.DataFrame(columns=required_columns)
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=required_columns)
    missing = set(required_columns) - set(df.columns)
    if missing:
        raise RuntimeError(
            f"Checkpoint {path} has incompatible schema; missing {sorted(missing)}"
        )
    if dedupe_keys and not df.empty:
        missing_keys = set(dedupe_keys) - set(df.columns)
        if missing_keys:
            raise RuntimeError(
                f"Checkpoint {path} missing dedupe keys {sorted(missing_keys)}"
            )
        df = df.drop_duplicates(list(dedupe_keys), keep="last").reset_index(drop=True)
    return df


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def stable_hash_order(df, key_col, salt):
    out = df.copy()
    out["_stable_hash"] = out[key_col].astype(str).map(
        lambda x: hashlib.sha256((str(salt) + "\\0" + x).encode("utf-8")).hexdigest()
    )
    return out.sort_values("_stable_hash").drop(columns=["_stable_hash"]).reset_index(drop=True)

## 8 — Build a fresh dataset partition and explicitly exclude every v9-used ID

In [ ]:
from datasets import load_dataset

SNAPSHOT_CSV = RESULTS / "benchmark_snapshot.csv"
SNAPSHOT_MANIFEST = RESULTS / "benchmark_snapshot_manifest.json"
V9_EXCLUSION_PATH = RESULTS / "v9_reconstructed_exclusion_ids.json"


def gsm8k_prompt(question):
    return (
        "Solve the following math problem. Show your reasoning, then end with "
        "a final line exactly in the form '#### <answer>'.\n\nProblem: "
        + str(question).strip()
    )


def mbpp_prompt(text):
    return (
        "Write Python code that solves the following task. Return only the code.\n\nTask: "
        + str(text).strip()
    )


def gsm_id(question):
    q = str(question).strip()
    return "gsm8k_" + hashlib.sha256(q.encode("utf-8")).hexdigest()[:16]


def mbpp_id(task_id):
    return f"mbpp_{int(task_id)}"


def reconstruct_v9_used_ids(gsm_train, gsm_test, mbpp_train, mbpp_validation, mbpp_test):
    v9_gsm_train = stable_hash_order(
        gsm_train, "question", f"gsm_train_{V9_DATA_SPLIT_SEED}"
    )
    v9_gsm_test = stable_hash_order(
        gsm_test, "question", f"gsm_test_{V9_DATA_SPLIT_SEED}"
    )
    v9_mbpp_train = stable_hash_order(
        mbpp_train, "task_id", f"mbpp_train_{V9_DATA_SPLIT_SEED}"
    )
    v9_mbpp_validation = stable_hash_order(
        mbpp_validation, "task_id", f"mbpp_val_{V9_DATA_SPLIT_SEED}"
    )
    v9_mbpp_test = stable_hash_order(
        mbpp_test, "task_id", f"mbpp_test_{V9_DATA_SPLIT_SEED}"
    )

    v9_train_target_n = (
        V9_SELECTION_TARGET_N
        + V9_TRAIN_TARGET_N
        + V9_VALIDATION_TARGET_N
    )

    return {
        "gsm_train": {
            gsm_id(q)
            for q in v9_gsm_train.iloc[:v9_train_target_n]["question"].tolist()
        },
        "gsm_test": {
            gsm_id(q)
            for q in v9_gsm_test.iloc[:V9_FINAL_TARGET_N]["question"].tolist()
        },
        "mbpp_train": {
            mbpp_id(x)
            for x in v9_mbpp_train.iloc[:V9_SELECTION_CONTROL_N]["task_id"].tolist()
        },
        "mbpp_validation": {
            mbpp_id(x)
            for x in v9_mbpp_validation.iloc[:V9_VALIDATION_CONTROL_N]["task_id"].tolist()
        },
        "mbpp_test": {
            mbpp_id(x)
            for x in v9_mbpp_test.iloc[:V9_FINAL_CONTROL_N]["task_id"].tolist()
        },
    }



def load_or_reconstruct_v9_used_ids(
    gsm_train,
    gsm_test,
    mbpp_train,
    mbpp_validation,
    mbpp_test,
):
    if V9_SNAPSHOT_CANDIDATE.exists():
        exact = pd.read_csv(
            V9_SNAPSHOT_CANDIDATE
        )

        required = {
            "example_id",
            "split",
            "kind",
            "source",
        }

        missing = (
            required
            - set(
                exact.columns
            )
        )

        if missing:
            raise RuntimeError(
                "Existing v9 snapshot is incompatible; missing "
                + repr(
                    sorted(
                        missing
                    )
                )
            )

        expected = {
            ("selection", "target"): V9_SELECTION_TARGET_N,
            ("train", "target"): V9_TRAIN_TARGET_N,
            ("validation", "target"): V9_VALIDATION_TARGET_N,
            ("final", "target"): V9_FINAL_TARGET_N,
            ("selection", "control"): V9_SELECTION_CONTROL_N,
            ("validation", "control"): V9_VALIDATION_CONTROL_N,
            ("final", "control"): V9_FINAL_CONTROL_N,
        }

        actual = (
            exact.groupby(
                [
                    "split",
                    "kind",
                ]
            )
            .size()
            .to_dict()
        )

        for key, value in expected.items():
            if int(
                actual.get(
                    key,
                    0,
                )
            ) != int(
                value
            ):
                raise RuntimeError(
                    "Existing v9 snapshot count mismatch for "
                    f"{key}: {actual.get(key, 0)} != {value}"
                )

        used = {
            "gsm_train": set(
                exact[
                    (
                        exact[
                            "kind"
                        ]
                        == "target"
                    )
                    & (
                        exact[
                            "split"
                        ]
                        != "final"
                    )
                ][
                    "example_id"
                ].astype(str)
            ),
            "gsm_test": set(
                exact[
                    (
                        exact[
                            "kind"
                        ]
                        == "target"
                    )
                    & (
                        exact[
                            "split"
                        ]
                        == "final"
                    )
                ][
                    "example_id"
                ].astype(str)
            ),
            "mbpp_train": set(
                exact[
                    (
                        exact[
                            "kind"
                        ]
                        == "control"
                    )
                    & (
                        exact[
                            "split"
                        ]
                        == "selection"
                    )
                ][
                    "example_id"
                ].astype(str)
            ),
            "mbpp_validation": set(
                exact[
                    (
                        exact[
                            "kind"
                        ]
                        == "control"
                    )
                    & (
                        exact[
                            "split"
                        ]
                        == "validation"
                    )
                ][
                    "example_id"
                ].astype(str)
            ),
            "mbpp_test": set(
                exact[
                    (
                        exact[
                            "kind"
                        ]
                        == "control"
                    )
                    & (
                        exact[
                            "split"
                        ]
                        == "final"
                    )
                ][
                    "example_id"
                ].astype(str)
            ),
        }

        snapshot_sha = sha256_file(
            V9_SNAPSHOT_CANDIDATE
        )

        if (
            snapshot_sha
            != EXPECTED_COMPLETED_V9_SNAPSHOT_SHA256
        ):
            print(
                "WARNING: local v9 snapshot SHA differs from the completed "
                "notebook captured in this project. Exact local IDs will still "
                "be excluded, which is safer than guessing."
            )

        metadata = {
            "source": "exact_local_v9_snapshot",
            "source_path": str(
                V9_SNAPSHOT_CANDIDATE
            ),
            "source_snapshot_sha256": snapshot_sha,
        }

        return used, metadata

    used = reconstruct_v9_used_ids(
        gsm_train,
        gsm_test,
        mbpp_train,
        mbpp_validation,
        mbpp_test,
    )

    metadata = {
        "source": "deterministic_v9_protocol_reconstruction",
        "source_path": None,
        "source_snapshot_sha256": None,
        "expected_completed_v9_snapshot_sha256": (
            EXPECTED_COMPLETED_V9_SNAPSHOT_SHA256
        ),
    }

    return used, metadata


def build_v10_snapshot():
    print("Loading pinned GSM8K...")
    gsm = load_dataset(
        TARGET_DATASET_ID,
        TARGET_DATASET_CONFIG,
        revision=TARGET_DATASET_REVISION,
    )
    if not {"train", "test"}.issubset(gsm.keys()):
        raise RuntimeError("GSM8K train/test splits missing.")

    gsm_train = pd.DataFrame(gsm["train"])
    gsm_test = pd.DataFrame(gsm["test"])

    if not {"question", "answer"}.issubset(gsm_train.columns):
        raise RuntimeError("GSM8K question/answer columns missing.")

    print("Loading pinned MBPP...")
    mbpp = load_dataset(
        CONTROL_DATASET_ID,
        CONTROL_DATASET_CONFIG,
        revision=CONTROL_DATASET_REVISION,
    )
    if not {"train", "validation", "test"}.issubset(mbpp.keys()):
        raise RuntimeError("MBPP full config train/validation/test splits missing.")

    mbpp_train = pd.DataFrame(mbpp["train"])
    mbpp_validation = pd.DataFrame(mbpp["validation"])
    mbpp_test = pd.DataFrame(mbpp["test"])

    for name, part in [
        ("train", mbpp_train),
        ("validation", mbpp_validation),
        ("test", mbpp_test),
    ]:
        if not {"task_id", "text", "code"}.issubset(part.columns):
            raise RuntimeError(f"MBPP {name} missing task_id/text/code columns.")

    v9_used, v9_exclusion_metadata = (
        load_or_reconstruct_v9_used_ids(
            gsm_train,
            gsm_test,
            mbpp_train,
            mbpp_validation,
            mbpp_test,
        )
    )

    exclusion_payload = {
        "metadata": v9_exclusion_metadata,
        "ids": {
            key: sorted(values)
            for key, values in v9_used.items()
        },
    }

    atomic_write_text(
        V9_EXCLUSION_PATH,
        json.dumps(
            exclusion_payload,
            indent=2,
        ),
    )

    # Exclude every v9-used target train/test ID.
    gsm_train = gsm_train[
        ~gsm_train["question"].map(gsm_id).isin(v9_used["gsm_train"])
    ].reset_index(drop=True)

    gsm_test = gsm_test[
        ~gsm_test["question"].map(gsm_id).isin(v9_used["gsm_test"])
    ].reset_index(drop=True)

    # v10 uses MBPP train for selection + both calibration controls,
    # and MBPP test only for the fresh final control.
    mbpp_train = mbpp_train[
        ~mbpp_train["task_id"].map(mbpp_id).isin(v9_used["mbpp_train"])
    ].reset_index(drop=True)

    mbpp_test = mbpp_test[
        ~mbpp_test["task_id"].map(mbpp_id).isin(v9_used["mbpp_test"])
    ].reset_index(drop=True)

    # Re-randomize the remaining pools with a new deterministic salt.
    gsm_train = stable_hash_order(
        gsm_train, "question", f"v10_gsm_train_{DATA_SPLIT_SEED}"
    )
    gsm_test = stable_hash_order(
        gsm_test, "question", f"v10_gsm_test_{DATA_SPLIT_SEED}"
    )
    mbpp_train = stable_hash_order(
        mbpp_train, "task_id", f"v10_mbpp_train_{DATA_SPLIT_SEED}"
    )
    mbpp_test = stable_hash_order(
        mbpp_test, "task_id", f"v10_mbpp_test_{DATA_SPLIT_SEED}"
    )

    required_gsm_train = (
        SELECTION_TARGET_N
        + TRAIN_TARGET_N
        + LR_CAL_TARGET_N
        + DOSE_CAL_TARGET_N
    )
    required_mbpp_train = (
        SELECTION_CONTROL_N
        + LR_CAL_CONTROL_N
        + DOSE_CAL_CONTROL_N
    )

    if len(gsm_train) < required_gsm_train:
        raise RuntimeError("Insufficient fresh GSM8K train rows.")
    if len(gsm_test) < FINAL_TARGET_N:
        raise RuntimeError("Insufficient fresh GSM8K test rows.")
    if len(mbpp_train) < required_mbpp_train:
        raise RuntimeError("Insufficient fresh MBPP train control rows.")
    if len(mbpp_test) < FINAL_CONTROL_N:
        raise RuntimeError("Insufficient fresh MBPP test control rows.")

    a = 0
    gsm_selection = gsm_train.iloc[a:a+SELECTION_TARGET_N].copy()
    a += SELECTION_TARGET_N
    gsm_fit = gsm_train.iloc[a:a+TRAIN_TARGET_N].copy()
    a += TRAIN_TARGET_N
    gsm_lr_cal = gsm_train.iloc[a:a+LR_CAL_TARGET_N].copy()
    a += LR_CAL_TARGET_N
    gsm_dose_cal = gsm_train.iloc[a:a+DOSE_CAL_TARGET_N].copy()
    gsm_final = gsm_test.iloc[:FINAL_TARGET_N].copy()

    b = 0
    mbpp_selection = mbpp_train.iloc[b:b+SELECTION_CONTROL_N].copy()
    b += SELECTION_CONTROL_N
    mbpp_lr_cal = mbpp_train.iloc[b:b+LR_CAL_CONTROL_N].copy()
    b += LR_CAL_CONTROL_N
    mbpp_dose_cal = mbpp_train.iloc[b:b+DOSE_CAL_CONTROL_N].copy()
    mbpp_final = mbpp_test.iloc[:FINAL_CONTROL_N].copy()

    rows = []

    def add_gsm(part, split):
        for r in part.itertuples(index=False):
            q = str(r.question).strip()
            answer = str(r.answer).strip()
            rows.append({
                "example_id": gsm_id(q),
                "split": split,
                "kind": "target",
                "source": TARGET_DATASET_ID,
                "prompt": gsm8k_prompt(q),
                "reference": answer,
                "gold_answer": answer.split("####")[-1].strip(),
            })

    def add_mbpp(part, split):
        for r in part.itertuples(index=False):
            rows.append({
                "example_id": mbpp_id(r.task_id),
                "split": split,
                "kind": "control",
                "source": CONTROL_DATASET_ID,
                "prompt": mbpp_prompt(r.text),
                "reference": (
                    str(r.code)
                    .replace("\r\n", "\n")
                    .replace("\r", "\n")
                    .strip()
                ),
                "gold_answer": "",
            })

    add_gsm(gsm_selection, "selection")
    add_gsm(gsm_fit, "train")
    add_gsm(gsm_lr_cal, "lr_calibration")
    add_gsm(gsm_dose_cal, "dose_calibration")
    add_gsm(gsm_final, "final")

    add_mbpp(mbpp_selection, "selection")
    add_mbpp(mbpp_lr_cal, "lr_calibration")
    add_mbpp(mbpp_dose_cal, "dose_calibration")
    add_mbpp(mbpp_final, "final")

    snapshot = pd.DataFrame(rows)

    all_v9_ids = set().union(*v9_used.values())
    overlap = set(snapshot["example_id"]) & all_v9_ids
    if overlap:
        raise RuntimeError(
            "v10 snapshot reuses v9 examples: "
            + repr(sorted(overlap)[:12])
        )

    return (
        snapshot,
        v9_used,
        v9_exclusion_metadata,
    )


if SNAPSHOT_CSV.exists():
    benchmark_df = pd.read_csv(SNAPSHOT_CSV)
    print("Reusing frozen v10 snapshot:", SNAPSHOT_CSV)

    if not V9_EXCLUSION_PATH.exists():
        raise RuntimeError(
            "Snapshot exists but v9 exclusion manifest is missing. "
            "Use a fresh RESULTS_ROOT."
        )
    v9_used_payload = json.loads(
        V9_EXCLUSION_PATH.read_text()
    )

    if (
        "ids"
        not in v9_used_payload
        or "metadata"
        not in v9_used_payload
    ):
        raise RuntimeError(
            "v9 exclusion manifest has an old/incompatible schema. "
            "Use a fresh RESULTS_ROOT."
        )

    v9_used = {
        k: set(v)
        for k, v in (
            v9_used_payload[
                "ids"
            ].items()
        )
    }

    v9_exclusion_metadata = (
        v9_used_payload[
            "metadata"
        ]
    )
else:
    (
        benchmark_df,
        v9_used,
        v9_exclusion_metadata,
    ) = build_v10_snapshot()

    atomic_to_csv(
        benchmark_df,
        SNAPSHOT_CSV,
        index=False,
    )

required_cols = {
    "example_id", "split", "kind", "source",
    "prompt", "reference", "gold_answer",
}
missing = required_cols - set(benchmark_df.columns)
if missing:
    raise RuntimeError(f"Snapshot missing columns: {sorted(missing)}")

benchmark_df["gold_answer"] = benchmark_df["gold_answer"].fillna("").astype(str)

expected_counts = {
    ("selection", "target"): SELECTION_TARGET_N,
    ("selection", "control"): SELECTION_CONTROL_N,
    ("train", "target"): TRAIN_TARGET_N,
    ("lr_calibration", "target"): LR_CAL_TARGET_N,
    ("lr_calibration", "control"): LR_CAL_CONTROL_N,
    ("dose_calibration", "target"): DOSE_CAL_TARGET_N,
    ("dose_calibration", "control"): DOSE_CAL_CONTROL_N,
    ("final", "target"): FINAL_TARGET_N,
    ("final", "control"): FINAL_CONTROL_N,
}
actual_counts = benchmark_df.groupby(["split", "kind"]).size().to_dict()
for key, expected in expected_counts.items():
    if int(actual_counts.get(key, 0)) != int(expected):
        raise RuntimeError(
            f"Snapshot count mismatch {key}: "
            f"{actual_counts.get(key, 0)} != {expected}"
        )

if benchmark_df["example_id"].duplicated().any():
    raise RuntimeError("Duplicate benchmark example IDs.")
if benchmark_df[["prompt", "reference"]].isna().any().any():
    raise RuntimeError("Null prompt/reference in v10 snapshot.")

all_v9_ids = set().union(*v9_used.values())
reused = set(benchmark_df["example_id"]) & all_v9_ids
if reused:
    raise RuntimeError("Freshness audit failed: v9 IDs appear in v10.")

# Every v10 protocol split must also be mutually disjoint.
split_sets = {
    split: set(part["example_id"])
    for split, part in benchmark_df.groupby("split")
}
for a, sa in split_sets.items():
    for b, sb in split_sets.items():
        if a < b and sa & sb:
            raise RuntimeError(f"v10 leakage between {a} and {b}.")

SNAPSHOT_SHA256 = sha256_file(SNAPSHOT_CSV)
V9_EXCLUSION_SHA256 = sha256_file(V9_EXCLUSION_PATH)

manifest = {
    "target_dataset_id": TARGET_DATASET_ID,
    "target_dataset_config": TARGET_DATASET_CONFIG,
    "target_dataset_revision": TARGET_DATASET_REVISION,
    "control_dataset_id": CONTROL_DATASET_ID,
    "control_dataset_config": CONTROL_DATASET_CONFIG,
    "control_dataset_revision": CONTROL_DATASET_REVISION,
    "snapshot_csv": str(SNAPSHOT_CSV),
    "snapshot_sha256": SNAPSHOT_SHA256,
    "v9_exclusion_sha256": V9_EXCLUSION_SHA256,
    "v9_exclusion_metadata": v9_exclusion_metadata,
    "counts": {f"{k[0]}:{k[1]}": int(v) for k, v in expected_counts.items()},
    "v9_overlap_count": 0,
}
atomic_write_text(
    SNAPSHOT_MANIFEST,
    json.dumps(manifest, indent=2),
)

print("Fresh v10 benchmark snapshot: PASS")
print("Snapshot SHA256:", SNAPSHOT_SHA256)
print("v9 exclusion SHA256:", V9_EXCLUSION_SHA256)
print("v9/v10 overlap: 0")
display(
    benchmark_df.groupby(["split", "kind", "source"])
    .size()
    .rename("n")
    .reset_index()
)

## 9 — Freeze the complete v10 protocol fingerprint

In [ ]:
PROTOCOL_CONFIG = {
    "protocol_version": PROTOCOL_VERSION,
    "model_id": "poolside/Laguna-XS.2",
    "snapshot_sha256": SNAPSHOT_SHA256,
    "v9_exclusion_sha256": V9_EXCLUSION_SHA256,
    "target_dataset_revision": TARGET_DATASET_REVISION,
    "control_dataset_revision": CONTROL_DATASET_REVISION,
    "v9_data_split_seed": V9_DATA_SPLIT_SEED,
    "data_split_seed": DATA_SPLIT_SEED,
    "selection_target_n": SELECTION_TARGET_N,
    "selection_control_n": SELECTION_CONTROL_N,
    "train_target_n": TRAIN_TARGET_N,
    "lr_cal_target_n": LR_CAL_TARGET_N,
    "lr_cal_control_n": LR_CAL_CONTROL_N,
    "dose_cal_target_n": DOSE_CAL_TARGET_N,
    "dose_cal_control_n": DOSE_CAL_CONTROL_N,
    "final_target_n": FINAL_TARGET_N,
    "final_control_n": FINAL_CONTROL_N,
    "global_grad_probe_cases": GLOBAL_GRAD_PROBE_CASES,
    "precise_grad_cases": PRECISE_GRAD_CASES,
    "attention_grad_probe_cases": ATTENTION_GRAD_PROBE_CASES,
    "selector_top_n": SELECTOR_TOP_N,
    "expert_budget_k": EXPERT_BUDGET_K,
    "guided_lora_n_layers": GUIDED_LORA_N_LAYERS,
    "random_lora_n_layers": RANDOM_LORA_N_LAYERS,
    "random_layer_seed": RANDOM_LAYER_SEED,
    "lora_target_modules": LORA_TARGET_MODULES,
    "expert_lr_grid": EXPERT_LR_GRID,
    "lora_lr_grid": LORA_LR_GRID,
    "lr_probe_updates": LR_PROBE_UPDATES,
    "lr_calibration_seed": LR_CALIBRATION_SEED,
    "dose_checkpoints": DOSE_CHECKPOINTS,
    "dose_calibration_seed": DOSE_CALIBRATION_SEED,
    "final_order_seeds": FINAL_ORDER_SEEDS,
    "optimizer_betas": OPTIMIZER_BETAS,
    "optimizer_weight_decay": OPTIMIZER_WEIGHT_DECAY,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "lora_alpha_multiplier": LORA_ALPHA_MULTIPLIER,
    "train_epochs": TRAIN_EPOCHS,
    "train_grad_accum": TRAIN_GRAD_ACCUM,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "generation_max_new_tokens": GENERATION_MAX_NEW_TOKENS,
    "primary_methods": PRIMARY_METHODS,
    "behavior_selection_rule": [
        "maximize generation_accuracy",
        "minimize abs(control_improvement)",
        "maximize target_improvement",
        "minimize updates_or_lr",
    ],
}
PROTOCOL_HASH = hashlib.sha256(
    json.dumps(PROTOCOL_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

protocol_path = RESULTS / "protocol_config.json"
payload = {
    "protocol_hash": PROTOCOL_HASH,
    "protocol_config": PROTOCOL_CONFIG,
}
if protocol_path.exists():
    existing = json.loads(protocol_path.read_text())
    if existing.get("protocol_hash") != PROTOCOL_HASH:
        raise RuntimeError(
            "Existing v10 result root has a different protocol hash. "
            "Use a fresh RESULTS_ROOT."
        )
else:
    atomic_write_text(protocol_path, json.dumps(payload, indent=2))

print("Protocol hash:", PROTOCOL_HASH)

## 10 — Resolve official BF16 Laguna XS.2 checkpoint

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2"

default_model_path = (
    Path("/home/ec2-user/workspace/models/Laguna-XS.2")
    if Path("/home/ec2-user/workspace").exists()
    else WORK_ROOT / "models" / "Laguna-XS.2"
)

MODEL_PATH = Path(
    os.environ.get("LAGUNA_BF16_PATH", str(default_model_path))
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00014.safetensors"
    for i in range(1, 15)
]

complete = (
    (MODEL_PATH / "config.json").exists()
    and all((MODEL_PATH / x).exists() for x in EXPECTED_SHARDS)
)

if not complete:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)

    free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30

    if free_gib < 85:
        raise RuntimeError(
            f"Need ~85 GiB free for a fresh BF16 download; found {free_gib:.1f} GiB."
        )

    print("Downloading Laguna XS.2 BF16...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_PATH),
        allow_patterns=[
            "*.safetensors",
            "*.json",
            "*.py",
            "*.jinja",
            "LICENSE*",
            "README*",
        ],
        max_workers=2,
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]

if missing:
    raise RuntimeError(f"Incomplete checkpoint; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]

print("MODEL_PATH:", MODEL_PATH)
print(f"14-shard payload: {sum(n for _, n in sizes)/1e9:.3f} GB")
print("Checkpoint verification: PASS")

## 11 — Register Laguna checkpoint conversion mapping

In [ ]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "Transformers does not expose the native Laguna checkpoint conversion mapping."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError("Laguna conversion mapping registration failed.")

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))

## 12 — Load BF16 model directly onto RTX PRO 6000

In [ ]:
import gc
import time
import torch
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(min(6, os.cpu_count() or 6))

try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad_(False)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical = [
    k for k in (missing + unexpected)
    if (
        ".mlp.experts" in k
        or ".mlp.gate" in k
        or "e_score_correction_bias" in k
    )
]

if critical or mismatched:
    raise RuntimeError(
        "Critical checkpoint mismatch.\n"
        f"critical sample: {critical[:12]}\n"
        f"mismatched sample: {mismatched[:12]}"
    )

del loading_info
gc.collect()
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()

print(f"Loaded in {(time.time()-t0)/60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free:   {free_b/2**30:.2f} GiB")
print(f"Host RAM available: {psutil.virtual_memory().available/2**30:.2f} GiB")
if hasattr(
    model,
    "get_experts_implementation",
):
    try:
        print(
            "Experts backend:",
            model.get_experts_implementation(),
        )
    except Exception as exc:
        print(
            "Experts backend query unavailable:",
            repr(
                exc
            ),
        )

if any(
    p.requires_grad
    for p in model.parameters()
):
    raise RuntimeError(
        "Base model unexpectedly has trainable parameters after load."
    )

print("BF16 load: PASS")

## 13 — Validate Laguna MoE architecture

In [ ]:
cfg = model.config

SPARSE_LAYERS = []

for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)

    if (
        mlp is not None
        and hasattr(mlp, "gate")
        and hasattr(mlp, "experts")
        and hasattr(mlp.experts, "gate_up_proj")
        and hasattr(mlp.experts, "down_proj")
    ):
        SPARSE_LAYERS.append(idx)

assert cfg.hidden_size == 2048
assert cfg.num_hidden_layers == 40
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert cfg.moe_intermediate_size == 512
assert len(SPARSE_LAYERS) == 39

sample_mlp = model.model.layers[SPARSE_LAYERS[0]].mlp

assert tuple(sample_mlp.experts.gate_up_proj.shape) == (256, 1024, 2048)
assert tuple(sample_mlp.experts.down_proj.shape) == (256, 2048, 512)

params_per_expert = (
    sample_mlp.experts.gate_up_proj[0].numel()
    + sample_mlp.experts.down_proj[0].numel()
)

print("Sparse layers:", SPARSE_LAYERS)
print(f"Params / expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")
print("Architecture validation: PASS")

## 14 — PEFT integration preflight

In [ ]:
from peft import LoraConfig, TaskType
if not hasattr(model, "add_adapter") or not hasattr(model, "delete_adapter"):
    raise RuntimeError("Transformers PEFT adapter integration unavailable.")
for p in model.parameters():
    p.requires_grad_(False)
print("PEFT integration import: PASS")

## 15 — Correct Laguna teacher forcing

In [ ]:
def chat_prefix_text(prompt):
    messages = [{"role": "user", "content": prompt}]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def parse_case(prompt, reference):
    prefix_text = chat_prefix_text(prompt)

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    start = 0

    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise RuntimeError(
            f"Could not identify answer boundary for: {prompt!r}"
        )

    return full_ids, start

## 16 — Generic aligned scoring

In [ ]:
def build_scoring_batch(df):
    df = df.reset_index(drop=True).copy()

    parsed = [
        parse_case(r.prompt, r.reference)
        for r in df.itertuples(index=False)
    ]

    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, seq_len),
        dtype=torch.long,
    )

    targets = torch.full(
        (batch_size, max_ref),
        -100,
        dtype=torch.long,
    )

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1

        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "df": df,
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

# `logits_to_keep` is intentionally used here for efficient aligned
# reference-token scoring. On BF16 hardware it need not be bit-identical
# to computing the LM head over the entire sequence and slicing afterward,
# because those are different GEMM shapes. The dedicated sanity cell checks
# POSITION alignment and NLL agreement instead of raw-logit bit equality.
@torch.inference_mode()
def score_batch(batch):
    import torch.nn.functional as F

    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    per_example = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example

    return result

def assert_teacher_forcing_alignment(
    batch,
    label="batch",
):
    """
    Exact, model-independent off-by-one check.

    For every valid target token y_j, pred_positions[j] must be the input
    position immediately before y_j. This is the actual causal-LM alignment
    invariant; it does not depend on BF16 logit bit-equality.
    """
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    targets = batch["targets"]
    pred_positions = batch["pred_positions"]

    if pred_positions.ndim != 1:
        raise RuntimeError(
            f"{label}: pred_positions must be 1D."
        )

    if targets.shape[1] != pred_positions.numel():
        raise RuntimeError(
            f"{label}: target/pred-position length mismatch: "
            f"{targets.shape[1]} vs {pred_positions.numel()}"
        )

    for b in range(targets.shape[0]):
        valid = targets[b].ne(-100)
        pred = pred_positions[valid]
        target_pos = pred + 1

        if target_pos.numel() == 0:
            raise RuntimeError(
                f"{label}: row {b} has no valid reference tokens."
            )

        if int(target_pos.max().item()) >= input_ids.shape[1]:
            raise RuntimeError(
                f"{label}: row {b} target position exceeds sequence."
            )

        actual_target_tokens = input_ids[
            b,
            target_pos,
        ]

        expected_target_tokens = targets[
            b,
            valid,
        ]

        if not torch.equal(
            actual_target_tokens,
            expected_target_tokens,
        ):
            mismatch = (
                actual_target_tokens
                != expected_target_tokens
            ).nonzero(
                as_tuple=False
            )[0, 0].item()

            raise RuntimeError(
                f"{label}: teacher-forcing token alignment failed "
                f"for row {b}, target index {mismatch}."
            )

        if not bool(
            attention_mask[
                b,
                pred,
            ].bool().all().item()
        ):
            raise RuntimeError(
                f"{label}: prediction position lands on padding in row {b}."
            )

        if not bool(
            attention_mask[
                b,
                target_pos,
            ].bool().all().item()
        ):
            raise RuntimeError(
                f"{label}: target token lands on padding in row {b}."
            )

    return True

## 17 — Chunked scoring helpers for larger real benchmark splits

In [ ]:
def build_scoring_batches(df, batch_size=8):
    df = df.reset_index(drop=True).copy()
    return [
        build_scoring_batch(df.iloc[start:start+int(batch_size)])
        for start in range(0, len(df), int(batch_size))
    ]


def score_batches_detailed(batches):
    out = {"nll": [], "token_acc": [], "seq_exact": []}
    for batch in batches:
        d = score_batch_detailed(batch)
        for key in out:
            out[key].append(np.asarray(d[key]))
    return {
        key: np.concatenate(parts) if parts else np.empty(0, dtype=np.float64)
        for key, parts in out.items()
    }


def evaluate_df_against_base(df, base_detail, batch_size=8):
    df = df.reset_index(drop=True).copy()
    batches = build_scoring_batches(df, batch_size=batch_size)
    detail = score_batches_detailed(batches)
    target_mask = (df["kind"].values == "target")
    control_mask = (df["kind"].values == "control")
    if not target_mask.any() or not control_mask.any():
        raise RuntimeError("Evaluation dataframe must contain both target and control rows.")

    target_nll = float(detail["nll"][target_mask].mean())
    control_nll = float(detail["nll"][control_mask].mean())
    base_target_nll = float(np.asarray(base_detail["nll"])[target_mask].mean())
    base_control_nll = float(np.asarray(base_detail["nll"])[control_mask].mean())

    target_improvement = base_target_nll - target_nll
    control_improvement = base_control_nll - control_nll
    control_damage = -control_improvement
    utility_score = target_improvement - CONTROL_PENALTY * max(control_damage, 0.0)
    specific_gain = target_improvement - control_improvement

    relative_target_improvement = target_improvement / max(base_target_nll, 1e-12)
    relative_control_improvement = control_improvement / max(base_control_nll, 1e-12)
    relative_control_damage = -relative_control_improvement
    relative_utility_score = (
        relative_target_improvement
        - CONTROL_PENALTY * max(relative_control_damage, 0.0)
    )
    relative_specific_gain = relative_target_improvement - relative_control_improvement

    result = {
        "base_target_nll": base_target_nll,
        "base_control_nll": base_control_nll,
        "target_nll": target_nll,
        "control_nll": control_nll,
        "target_improvement": float(target_improvement),
        "control_improvement": float(control_improvement),
        "control_damage": float(control_damage),
        "control_abs_shift": float(abs(control_improvement)),
        "utility_score": float(utility_score),
        "specific_gain": float(specific_gain),
        "selective_target_gain": float(
            target_improvement - abs(control_improvement)
        ),
        "relative_target_improvement": float(relative_target_improvement),
        "relative_control_improvement": float(relative_control_improvement),
        "relative_control_damage": float(relative_control_damage),
        "relative_utility_score": float(relative_utility_score),
        "relative_specific_gain": float(relative_specific_gain),
        "target_token_acc": float(detail["token_acc"][target_mask].mean()),
        "control_token_acc": float(detail["token_acc"][control_mask].mean()),
        "target_seq_exact": float(detail["seq_exact"][target_mask].mean()),
        "control_seq_exact": float(detail["seq_exact"][control_mask].mean()),
        "per_example_nll": detail["nll"],
    }

    del batches
    gc.collect(); torch.cuda.empty_cache()
    return result

# Core fixed-routing, routing, gradient and exact-baseline surgery helpers

In [ ]:
from contextlib import contextmanager
import types

def get_sparse_mlp(layer_idx):
    layer_idx = int(layer_idx)

    if layer_idx not in SPARSE_LAYERS:
        raise ValueError(f"Layer {layer_idx} is not sparse.")

    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(
    layer_idx,
    expert_ids=None,
    zero_all_routed=False,
    renormalize=False,
):
    mlp = get_sparse_mlp(layer_idx)
    gate = mlp.gate
    original_forward = gate.forward

    expert_ids = (
        []
        if expert_ids is None
        else [int(x) for x in expert_ids]
    )

    def patched_forward(self, hidden_states):
        router_logits, routing_weights, selected_experts = (
            original_forward(hidden_states)
        )

        if zero_all_routed:
            routing_weights = torch.zeros_like(routing_weights)

        elif expert_ids:
            ids = torch.tensor(
                expert_ids,
                device=selected_experts.device,
                dtype=selected_experts.dtype,
            )

            keep = ~torch.isin(selected_experts, ids)

            routing_weights = (
                routing_weights
                * keep.to(routing_weights.dtype)
            )

            if renormalize:
                denom = routing_weights.sum(
                    dim=-1,
                    keepdim=True,
                )

                routing_weights = torch.where(
                    denom > 0,
                    routing_weights
                    / denom.clamp_min(1e-12),
                    routing_weights,
                )

        return (
            router_logits,
            routing_weights,
            selected_experts,
        )

    gate.forward = types.MethodType(
        patched_forward,
        gate,
    )

    try:
        yield
    finally:
        gate.forward = original_forward

In [ ]:
def summarize_selection_delta(ablated_nll):
    delta = (
        np.asarray(ablated_nll)
        - SELECTION_BASE_NLL
    )

    target_delta = float(
        delta[selection_target_mask].mean()
    )

    control_delta = float(
        delta[selection_control_mask].mean()
    )

    causal_specificity = (
        target_delta
        - CONTROL_PENALTY
        * max(control_delta, 0.0)
    )

    baseline_target = float(
        SELECTION_BASE_NLL[
            selection_target_mask
        ].mean()
    )

    baseline_control = float(
        SELECTION_BASE_NLL[
            selection_control_mask
        ].mean()
    )

    target_relative_delta = (
        target_delta
        / max(
            baseline_target,
            1e-12,
        )
    )

    control_relative_delta = (
        control_delta
        / max(
            baseline_control,
            1e-12,
        )
    )

    causal_relative_specificity = (
        target_relative_delta
        - CONTROL_PENALTY
        * max(
            control_relative_delta,
            0.0,
        )
    )

    return {
        "target_delta_nll": target_delta,
        "control_delta_nll": control_delta,
        "causal_specificity": causal_specificity,
        "target_relative_delta": target_relative_delta,
        "control_relative_delta": control_relative_delta,
        "causal_relative_specificity": causal_relative_specificity,
        "per_example_delta": delta,
    }

def intervention_score(
    layer_idx,
    expert_ids,
    renormalize=False,
):
    with gate_intervention(
        layer_idx,
        expert_ids=expert_ids,
        renormalize=renormalize,
    ):
        nll = score_batch(SELECTION_BATCH)

    return summarize_selection_delta(nll)

def bootstrap_specificity(
    per_example_delta,
    n_boot=5000,
    seed=123,
):
    rng = np.random.default_rng(seed)
    d = np.asarray(
        per_example_delta,
        dtype=np.float64,
    )

    target = d[selection_target_mask]
    control = d[selection_control_mask]

    vals = np.empty(
        int(n_boot),
        dtype=np.float64,
    )

    for i in range(int(n_boot)):
        t = rng.choice(
            target,
            size=len(target),
            replace=True,
        ).mean()

        c = rng.choice(
            control,
            size=len(control),
            replace=True,
        ).mean()

        vals[i] = (
            t
            - CONTROL_PENALTY
            * max(c, 0.0)
        )

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

In [ ]:
def make_training_case(
    prompt,
    reference,
    max_length=1024,
):
    prefix_text = chat_prefix_text(
        prompt
    )

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    if len(full_ids) > max_length:
        raise ValueError(
            f"{len(full_ids)} tokens > {max_length}"
        )

    start = 0

    for a, b in zip(
        prefix_ids,
        full_ids,
    ):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise ValueError(
            "Could not identify answer boundary."
        )

    input_ids = torch.tensor(
        full_ids,
        dtype=torch.long,
        device="cuda:0",
    ).unsqueeze(0)

    attention_mask = torch.ones_like(
        input_ids
    )

    pred_positions = torch.arange(
        start - 1,
        len(full_ids) - 1,
        dtype=torch.long,
        device="cuda:0",
    )

    targets = torch.tensor(
        full_ids[start:],
        dtype=torch.long,
        device="cuda:0",
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pred_positions": pred_positions,
        "targets": targets,
    }

In [ ]:
def make_supervised_position_mask(
    batch,
):
    mask = torch.zeros_like(
        batch[
            "attention_mask"
        ],
        dtype=torch.bool,
    )

    valid_targets = (
        batch[
            "targets"
        ]
        .ne(
            -100
        )
    )

    positions = batch[
        "pred_positions"
    ]

    for b in range(
        mask.shape[
            0
        ]
    ):
        valid = valid_targets[
            b
        ]

        pos = positions[
            valid
        ]

        mask[
            b,
            pos,
        ] = True

    return mask

@contextmanager
def capture_routing_with_mask(
    batch,
    token_mask,
):
    records = {}
    originals = []

    token_mask_flat = (
        token_mask
        .reshape(
            -1
        )
        .bool()
    )

    try:
        for layer_idx in (
            SPARSE_LAYERS
        ):
            gate = (
                get_sparse_mlp(
                    layer_idx
                )
                .gate
            )

            original = (
                gate.forward
            )

            originals.append(
                (
                    gate,
                    original,
                )
            )

            def make_forward(
                idx,
                original_forward,
            ):
                def patched(
                    self,
                    hidden_states,
                ):
                    (
                        logits,
                        weights,
                        selected,
                    ) = (
                        original_forward(
                            hidden_states
                        )
                    )

                    with torch.no_grad():
                        mask = (
                            token_mask_flat
                        )

                        if (
                            mask.numel()
                            != selected.shape[
                                0
                            ]
                        ):
                            raise RuntimeError(
                                f"Routing mask/token mismatch at layer {idx}: "
                                f"{mask.numel()} vs {selected.shape[0]}"
                            )

                        mask = mask.to(
                            selected.device
                        )

                        ids = (
                            selected[
                                mask
                            ]
                            .reshape(
                                -1
                            )
                            .long()
                        )

                        ws = (
                            weights[
                                mask
                            ]
                            .reshape(
                                -1
                            )
                            .float()
                        )

                        counts = (
                            torch.bincount(
                                ids,
                                minlength=(
                                    cfg.num_experts
                                ),
                            )
                        )

                        wsum = (
                            torch.zeros(
                                cfg.num_experts,
                                device=(
                                    ws.device
                                ),
                                dtype=(
                                    torch.float32
                                ),
                            )
                        )

                        wsum.scatter_add_(
                            0,
                            ids,
                            ws,
                        )

                        records[
                            int(
                                idx
                            )
                        ] = {
                            "tokens": int(
                                mask.sum()
                                .item()
                            ),
                            "counts": (
                                counts
                                .cpu()
                            ),
                            "weight_sums": (
                                wsum
                                .cpu()
                            ),
                        }

                    return (
                        logits,
                        weights,
                        selected,
                    )

                return patched

            gate.forward = (
                types.MethodType(
                    make_forward(
                        layer_idx,
                        original,
                    ),
                    gate,
                )
            )

        yield records

    finally:
        for (
            gate,
            original,
        ) in reversed(
            originals
        ):
            gate.forward = (
                original
            )

In [ ]:
def _fused_expert_grad_norms(
    gate_up_grad,
    down_grad,
    chunk_size=8,
):
    if gate_up_grad is None or down_grad is None:
        raise RuntimeError(
            "Expected fused expert gradients, got None."
        )

    n = gate_up_grad.shape[0]
    out = torch.empty(
        n,
        dtype=torch.float64,
        device="cpu",
    )

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)

        gu = gate_up_grad[start:end].float()
        down = down_grad[start:end].float()

        sq = (
            gu.square().sum(dim=(1, 2))
            + down.square().sum(dim=(1, 2))
        )

        out[start:end] = (
            torch.sqrt(sq)
            .double()
            .cpu()
        )

        del gu, down, sq

    return out.numpy()

def _gradient_objective_for_cases(
    cases,
    max_cases,
):
    used = min(
        int(max_cases),
        len(cases),
    )

    if used <= 0:
        raise RuntimeError("No gradient probe cases.")

    for case in cases[:used]:
        with torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ):
            out = model(
                input_ids=case["input_ids"],
                attention_mask=case["attention_mask"],
                use_cache=False,
                logits_to_keep=case["pred_positions"],
                return_dict=True,
            )

            logits = out.logits.float()

            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                case["targets"].reshape(-1),
            ) / used

        loss.backward()

        del out, logits, loss

def scan_layer_fused_gradients(
    layer_idx,
    target_cases,
    control_cases,
    max_cases=8,
):
    mlp = get_sparse_mlp(layer_idx)
    experts = mlp.experts

    gate_up = experts.gate_up_proj
    down = experts.down_proj

    if gate_up.dtype != torch.bfloat16 or down.dtype != torch.bfloat16:
        print(
            "WARNING: fused expert dtype is",
            gate_up.dtype,
            down.dtype,
        )

    for p in model.parameters():
        p.requires_grad_(False)

    gate_up.requires_grad_(True)
    down.requires_grad_(True)

    model.eval()

    try:
        gate_up.grad = None
        down.grad = None

        _gradient_objective_for_cases(
            target_cases,
            max_cases,
        )

        target_norm = _fused_expert_grad_norms(
            gate_up.grad,
            down.grad,
            GLOBAL_GRAD_CHUNK,
        )

        gate_up.grad = None
        down.grad = None
        torch.cuda.empty_cache()

        _gradient_objective_for_cases(
            control_cases,
            max_cases,
        )

        control_norm = _fused_expert_grad_norms(
            gate_up.grad,
            down.grad,
            GLOBAL_GRAD_CHUNK,
        )

        return target_norm, control_norm

    finally:
        gate_up.grad = None
        down.grad = None

        gate_up.requires_grad_(False)
        down.requires_grad_(False)

        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import types


class SurgicalExpertBank(nn.Module):
    """
    Exact-baseline delta surgery.

    Forward =
        original_frozen_expert_output
        + trainable_selected_expert_output
        - frozen_selected_expert_output

    At initialization:
        trainable_selected == frozen_selected

    therefore:
        delta == 0 exactly

    and the untouched model's original expert kernel remains the base path.
    """

    def __init__(self, selected_pairs):
        super().__init__()

        self.selected_pairs = sorted({
            (int(layer_idx), int(expert_id))
            for layer_idx, expert_id in selected_pairs
        })

        self.params = nn.ParameterDict()
        self.key_map = {}

        self.original_forwards = {}
        self.installed = False

        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            gu_key = f"L{layer_idx}_E{expert_id}_gu"
            down_key = f"L{layer_idx}_E{expert_id}_down"

            # FP32 master parameters.
            self.params[gu_key] = nn.Parameter(
                experts.gate_up_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.params[down_key] = nn.Parameter(
                experts.down_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.key_map[
                (layer_idx, expert_id)
            ] = (
                gu_key,
                down_key,
            )

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(
            p.numel()
            for p in self.parameters()
        )

    def install(self):
        if self.installed:
            return

        bank = self

        selected_layers = sorted({
            layer_idx
            for (
                layer_idx,
                _,
            ) in self.selected_pairs
        })

        try:
            for layer_idx in selected_layers:
                experts = get_sparse_mlp(
                    layer_idx
                ).experts

                original_forward = (
                    experts.forward
                )

                self.original_forwards[
                    layer_idx
                ] = original_forward

                selected_ids = sorted({
                    expert_id
                    for (
                        l,
                        expert_id,
                    ) in (
                        self.selected_pairs
                    )
                    if l
                    == layer_idx
                })

                def make_forward(
                    idx,
                    base_experts,
                    original_fn,
                    selected_expert_ids,
                ):
                    def patched_forward(
                        self_experts,
                        hidden_states,
                        top_k_index,
                        top_k_weights,
                    ):
                        # Preserve Laguna's original expert execution as the
                        # frozen base path.
                        base_output = (
                            original_fn(
                                hidden_states,
                                top_k_index,
                                top_k_weights,
                            )
                        )

                        correction = (
                            torch.zeros_like(
                                base_output
                            )
                        )

                        for expert_id in (
                            selected_expert_ids
                        ):
                            (
                                token_idx,
                                top_k_pos,
                            ) = torch.where(
                                top_k_index
                                == expert_id
                            )

                            if (
                                token_idx
                                .numel()
                                == 0
                            ):
                                continue

                            current_state = (
                                hidden_states[
                                    token_idx
                                ]
                            )

                            compute_dtype = (
                                current_state
                                .dtype
                            )

                            (
                                gu_key,
                                down_key,
                            ) = bank.key_map[
                                (
                                    idx,
                                    expert_id,
                                )
                            ]

                            train_gu = (
                                bank.params[
                                    gu_key
                                ]
                                .to(
                                    compute_dtype
                                )
                            )

                            train_down = (
                                bank.params[
                                    down_key
                                ]
                                .to(
                                    compute_dtype
                                )
                            )

                            frozen_gu = (
                                base_experts
                                .gate_up_proj[
                                    expert_id
                                ]
                                .detach()
                            )

                            frozen_down = (
                                base_experts
                                .down_proj[
                                    expert_id
                                ]
                                .detach()
                            )

                            (
                                train_gate,
                                train_up,
                            ) = F.linear(
                                current_state,
                                train_gu,
                            ).chunk(
                                2,
                                dim=-1,
                            )

                            train_hidden = (
                                base_experts
                                .act_fn(
                                    train_gate
                                )
                                * train_up
                            )

                            train_hidden = (
                                F.linear(
                                    train_hidden,
                                    train_down,
                                )
                            )

                            (
                                frozen_gate,
                                frozen_up,
                            ) = F.linear(
                                current_state,
                                frozen_gu,
                            ).chunk(
                                2,
                                dim=-1,
                            )

                            frozen_hidden = (
                                base_experts
                                .act_fn(
                                    frozen_gate
                                )
                                * frozen_up
                            )

                            frozen_hidden = (
                                F.linear(
                                    frozen_hidden,
                                    frozen_down,
                                )
                            )

                            route_weight = (
                                top_k_weights[
                                    token_idx,
                                    top_k_pos,
                                    None,
                                ]
                            )

                            delta = (
                                (
                                    train_hidden
                                    - frozen_hidden
                                )
                                * route_weight
                            ).to(
                                base_output.dtype
                            )

                            correction = (
                                correction
                                .index_add(
                                    0,
                                    token_idx,
                                    delta,
                                )
                            )

                        return (
                            base_output
                            + correction
                        )

                    return patched_forward

                experts.forward = (
                    types.MethodType(
                        make_forward(
                            layer_idx,
                            experts,
                            original_forward,
                            selected_ids,
                        ),
                        experts,
                    )
                )

            self.installed = True

        except Exception:
            # A partially installed multi-layer bank must never be allowed
            # to leak patched forwards into later arms.
            for (
                layer_idx,
                original_forward,
            ) in list(
                self.original_forwards
                .items()
            ):
                get_sparse_mlp(
                    layer_idx
                ).experts.forward = (
                    original_forward
                )

            self.original_forwards.clear()
            self.installed = False
            raise

    def restore(self):
        # Restore even after a partially failed install.
        for (
            layer_idx,
            original_forward,
        ) in list(
            self.original_forwards
            .items()
        ):
            get_sparse_mlp(
                layer_idx
            ).experts.forward = (
                original_forward
            )

        self.original_forwards.clear()
        self.installed = False

In [ ]:
def verify_routed_bank_equivalence_pairs(
    pairs,
    probe_df=None,
    tol=1e-5,
):
    pairs = sorted({
        tuple(
            map(
                int,
                pair,
            )
        )
        for pair in pairs
    })

    if not pairs:
        raise ValueError(
            "No expert pairs supplied for equivalence test."
        )

    if probe_df is None:
        probe_df = (
            selection_df
            .head(8)
        )

    batch = build_scoring_batch(
        probe_df
    )

    before = score_batch(
        batch
    )

    bank = SurgicalExpertBank(
        pairs
    )

    try:
        bank.install()

        after = score_batch(
            batch
        )

    finally:
        bank.restore()

        del bank
        del batch

        gc.collect()
        torch.cuda.empty_cache()

    diff = float(
        np.max(
            np.abs(
                before
                - after
            )
        )
    )

    print(
        "Routed bank equivalence",
        pairs,
        "max ΔNLL:",
        diff,
    )

    if diff > float(
        tol
    ):
        raise RuntimeError(
            "SurgicalExpertBank changes outputs before training."
        )

    return diff

def verify_routed_bank_equivalence(
    pair,
    probe_df=None,
    tol=1e-5,
):
    return (
        verify_routed_bank_equivalence_pairs(
            [
                pair
            ],
            probe_df=probe_df,
            tol=tol,
        )
    )

In [ ]:
def _configure_grad_checkpointing():
    if hasattr(
        model,
        "enable_input_require_grads",
    ):
        model.enable_input_require_grads()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )
    except TypeError:
        model.gradient_checkpointing_enable()

def _disable_grad_checkpointing():
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

    if hasattr(
        model,
        "disable_input_require_grads",
    ):
        model.disable_input_require_grads()

def _backprop_bank_cases(
    bank,
    cases,
    max_cases,
):
    used = min(
        int(max_cases),
        len(cases),
    )

    for p in bank.parameters():
        p.grad = None

    for case in cases[:used]:
        with torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ):
            out = model(
                input_ids=case["input_ids"],
                attention_mask=case["attention_mask"],
                use_cache=False,
                logits_to_keep=case["pred_positions"],
                return_dict=True,
            )

            logits = out.logits.float()

            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                case["targets"].reshape(-1),
            ) / used

        loss.backward()

        del out, logits, loss

    grads = []

    for p in bank.parameters():
        if p.grad is None:
            grads.append(
                torch.zeros_like(
                    p,
                    device="cpu",
                    dtype=torch.float32,
                )
            )
        else:
            grads.append(
                p.grad.detach().float().cpu().clone()
            )

    return grads

def precise_gradient_geometry(
    pair,
    target_cases,
    control_cases,
    max_cases=12,
):
    pair = tuple(
        map(
            int,
            pair,
        )
    )

    grad_seed = (
        910000
        + 10000
        * _capability_number(
            CURRENT_CAPABILITY
        )
        + pair[
            0
        ]
        * 257
        + pair[
            1
        ]
    )

    torch.manual_seed(
        grad_seed
    )

    torch.cuda.manual_seed_all(
        grad_seed
    )

    bank = SurgicalExpertBank(
        [
            pair
        ]
    )

    installed = False

    try:
        bank.install()
        installed = True

        for p in (
            model.parameters()
        ):
            p.requires_grad_(
                False
            )

        _configure_grad_checkpointing()

        model.train()
        bank.train()

        target_grads = (
            _backprop_bank_cases(
                bank,
                target_cases,
                max_cases,
            )
        )

        control_grads = (
            _backprop_bank_cases(
                bank,
                control_cases,
                max_cases,
            )
        )

        target_sq = 0.0
        control_sq = 0.0
        dot = 0.0

        for (
            gt,
            gc_,
        ) in zip(
            target_grads,
            control_grads,
        ):
            if not bool(
                torch.isfinite(
                    gt
                ).all()
            ):
                raise RuntimeError(
                    f"Non-finite target gradient for {pair}."
                )

            if not bool(
                torch.isfinite(
                    gc_
                ).all()
            ):
                raise RuntimeError(
                    f"Non-finite control gradient for {pair}."
                )

            target_sq += float(
                torch.sum(
                    gt
                    * gt
                ).item()
            )

            control_sq += float(
                torch.sum(
                    gc_
                    * gc_
                ).item()
            )

            dot += float(
                torch.sum(
                    gt
                    * gc_
                ).item()
            )

        target_l2 = float(
            np.sqrt(
                target_sq
            )
        )

        control_l2 = float(
            np.sqrt(
                control_sq
            )
        )

        cosine = float(
            dot
            / max(
                target_l2
                * control_l2,
                1e-12,
            )
        )

        if control_sq <= 1e-24:
            target_orthogonal_sq = target_sq
        else:
            target_orthogonal_sq = max(
                target_sq
                - (dot * dot) / control_sq,
                0.0,
            )

        target_orthogonal_l2 = float(
            np.sqrt(
                target_orthogonal_sq
            )
        )

        target_orthogonal_fraction = float(
            target_orthogonal_l2
            / max(
                target_l2,
                1e-12,
            )
        )

        return {
            "precise_target_grad_l2": (
                target_l2
            ),
            "precise_control_grad_l2": (
                control_l2
            ),
            "precise_grad_dot": dot,
            "precise_grad_cosine": (
                cosine
            ),
            "precise_grad_specific": (
                target_l2
                - CONTROL_PENALTY
                * control_l2
            ),
            "precise_grad_ratio": (
                target_l2
                / (
                    control_l2
                    + 1e-12
                )
            ),
            "precise_target_orthogonal_l2": (
                target_orthogonal_l2
            ),
            "precise_target_orthogonal_fraction": (
                target_orthogonal_fraction
            ),
            "precise_abs_grad_cosine": float(
                abs(
                    cosine
                )
            ),
        }

    finally:
        if installed:
            bank.restore()
        else:
            # restore() is robust to a partial failed installation.
            bank.restore()

        _disable_grad_checkpointing()

        model.eval()

        del bank

        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# Same selective reference-token path as score_batch.
@torch.inference_mode()
def score_batch_detailed(batch):
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    nll = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    preds = logits.argmax(dim=-1)

    correct = (
        preds.eq(targets)
        & valid
    )

    token_acc = (
        correct.sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    seq_exact = (
        (correct | ~valid)
        .all(dim=-1)
        .float()
    )

    result = {
        "nll": nll.cpu().numpy(),
        "token_acc": token_acc.cpu().numpy(),
        "seq_exact": seq_exact.cpu().numpy(),
    }

    del out, logits, losses, preds

    return result

def evaluate_batch_against_base(
    batch,
    df,
    base_detail,
):
    d = score_batch_detailed(batch)

    target_mask = (
        df["kind"].values == "target"
    )
    control_mask = (
        df["kind"].values == "control"
    )

    target_nll = float(
        d["nll"][target_mask].mean()
    )
    control_nll = float(
        d["nll"][control_mask].mean()
    )

    base_target_nll = float(
        base_detail["nll"][
            target_mask
        ].mean()
    )
    base_control_nll = float(
        base_detail["nll"][
            control_mask
        ].mean()
    )

    target_improvement = (
        base_target_nll - target_nll
    )

    control_improvement = (
        base_control_nll - control_nll
    )

    control_damage = (
        -control_improvement
    )

    utility_score = (
        target_improvement
        - CONTROL_PENALTY
        * max(control_damage, 0.0)
    )

    specific_gain = (
        target_improvement
        - control_improvement
    )

    relative_target_improvement = (
        target_improvement
        / max(
            base_target_nll,
            1e-12,
        )
    )

    relative_control_improvement = (
        control_improvement
        / max(
            base_control_nll,
            1e-12,
        )
    )

    relative_specific_gain = (
        relative_target_improvement
        - relative_control_improvement
    )

    return {
        "base_target_nll": base_target_nll,
        "base_control_nll": base_control_nll,
        "target_nll": target_nll,
        "control_nll": control_nll,
        "target_improvement": float(target_improvement),
        "control_improvement": float(control_improvement),
        "control_damage": float(control_damage),
        "utility_score": float(utility_score),
        "specific_gain": float(specific_gain),
        "relative_target_improvement": float(
            relative_target_improvement
        ),
        "relative_control_improvement": float(
            relative_control_improvement
        ),
        "relative_specific_gain": float(
            relative_specific_gain
        ),
        "target_token_acc": float(
            d["token_acc"][target_mask].mean()
        ),
        "control_token_acc": float(
            d["token_acc"][control_mask].mean()
        ),
        "target_seq_exact": float(
            d["seq_exact"][target_mask].mean()
        ),
        "control_seq_exact": float(
            d["seq_exact"][control_mask].mean()
        ),
        "per_example_nll": d["nll"],
    }

In [ ]:
def capture_routed_base_guard(pairs):
    guard = {}

    for layer_idx, expert_id in pairs:
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        guard[(layer_idx, expert_id)] = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu().clone(),
            experts.down_proj[
                expert_id
            ].detach().cpu().clone(),
        )

    return guard

def assert_routed_base_unchanged(guard):
    for (
        layer_idx,
        expert_id,
    ), (gu0, down0) in guard.items():
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        gu1 = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu()
        )

        down1 = (
            experts.down_proj[
                expert_id
            ].detach().cpu()
        )

        if not torch.equal(gu0, gu1):
            raise RuntimeError(
                f"Frozen base gate_up changed: "
                f"L{layer_idx}/E{expert_id}"
            )

        if not torch.equal(down0, down1):
            raise RuntimeError(
                f"Frozen base down changed: "
                f"L{layer_idx}/E{expert_id}"
            )

def state_l2(state):
    total = 0.0

    for v in state.values():
        x = v.detach().float()
        total += float(
            torch.sum(x * x).item()
        )

    return float(np.sqrt(total))

def state_delta_l2(before, after):
    total = 0.0

    for k in before:
        d = (
            after[k].detach().float()
            - before[k].detach().float()
        )
        total += float(
            torch.sum(d * d).item()
        )

    return float(np.sqrt(total))

In [ ]:
def restore_routed_base_guard(
    guard,
):
    with torch.no_grad():
        for (
            layer_idx,
            expert_id,
        ), (
            gu0,
            down0,
        ) in guard.items():
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            experts.gate_up_proj[
                expert_id
            ].copy_(
                gu0.to(
                    device=(
                        experts
                        .gate_up_proj
                        .device
                    ),
                    dtype=(
                        experts
                        .gate_up_proj
                        .dtype
                    ),
                )
            )

            experts.down_proj[
                expert_id
            ].copy_(
                down0.to(
                    device=(
                        experts
                        .down_proj
                        .device
                    ),
                    dtype=(
                        experts
                        .down_proj
                        .dtype
                    ),
                )
            )

def copy_bank_into_base(
    bank,
    pairs,
):
    with torch.no_grad():
        for (
            layer_idx,
            expert_id,
        ) in pairs:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            (
                gu_key,
                down_key,
            ) = bank.key_map[
                (
                    layer_idx,
                    expert_id,
                )
            ]

            experts.gate_up_proj[
                expert_id
            ].copy_(
                bank.params[
                    gu_key
                ].detach().to(
                    device=(
                        experts
                        .gate_up_proj
                        .device
                    ),
                    dtype=(
                        experts
                        .gate_up_proj
                        .dtype
                    ),
                )
            )

            experts.down_proj[
                expert_id
            ].copy_(
                bank.params[
                    down_key
                ].detach().to(
                    device=(
                        experts
                        .down_proj
                        .device
                    ),
                    dtype=(
                        experts
                        .down_proj
                        .dtype
                    ),
                )
            )

def verify_native_merge_roundtrip_pairs(
    pairs,
    probe_df=None,
    tol=1e-5,
):
    """
    Initial FP32 bank weights are exact lifts of BF16 expert slices.
    Casting them back to BF16 and merging must therefore be a no-op.
    """
    pairs = sorted({
        tuple(
            map(
                int,
                pair,
            )
        )
        for pair in pairs
    })

    if not pairs:
        raise ValueError(
            "No expert pairs supplied for merge roundtrip."
        )

    if probe_df is None:
        probe_df = (
            selection_df
            .head(8)
        )

    batch = build_scoring_batch(
        probe_df
    )

    before = score_batch(
        batch
    )

    guard = capture_routed_base_guard(
        pairs
    )

    bank = SurgicalExpertBank(
        pairs
    )

    try:
        copy_bank_into_base(
            bank,
            pairs,
        )

        # This is a stronger check than output equivalence: the initial FP32
        # masters came from the frozen BF16 slices, so casting them back to
        # BF16 must reproduce every selected tensor element exactly.
        for (
            layer_idx,
            expert_id,
        ), (
            gu0,
            down0,
        ) in guard.items():
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            gu1 = (
                experts.gate_up_proj[
                    expert_id
                ].detach().cpu()
            )

            down1 = (
                experts.down_proj[
                    expert_id
                ].detach().cpu()
            )

            if not torch.equal(
                gu0,
                gu1,
            ):
                raise RuntimeError(
                    f"Initial native merge changed gate_up values at "
                    f"L{layer_idx}/E{expert_id}."
                )

            if not torch.equal(
                down0,
                down1,
            ):
                raise RuntimeError(
                    f"Initial native merge changed down_proj values at "
                    f"L{layer_idx}/E{expert_id}."
                )

        after = score_batch(
            batch
        )

        diff = float(
            np.max(
                np.abs(
                    before
                    - after
                )
            )
        )

    finally:
        restore_routed_base_guard(
            guard
        )

        assert_routed_base_unchanged(
            guard
        )

        del bank
        del guard
        del batch

        gc.collect()
        torch.cuda.empty_cache()

    print(
        "Native merge roundtrip",
        pairs,
        "max ΔNLL:",
        diff,
    )

    if diff > float(
        tol
    ):
        raise RuntimeError(
            "Initial bank -> BF16 native merge is not an exact no-op."
        )

    return diff

def verify_native_merge_roundtrip(
    pair,
    probe_df=None,
    tol=1e-5,
):
    return (
        verify_native_merge_roundtrip_pairs(
            [
                pair
            ],
            probe_df=probe_df,
            tol=tol,
        )
    )

def train_routed_experts(
    selector,
    pairs,
    order_seed,
    lr,
    epochs,
    max_updates,
    grad_accum,
    eval_batch,
    eval_df,
    eval_base_detail,
):
    """
    Train selected experts through the exact-baseline delta path.

    Two post-training evaluations are recorded:
      * adapter path (primary, comparable to v6/v7);
      * native BF16 merge (secondary deployment/quantization audit).

    Gradient accumulation is averaged by the ACTUAL number of examples in
    each accumulation window, so a final partial window is not underweighted.
    """
    pairs = sorted({
        tuple(
            map(
                int,
                pair,
            )
        )
        for pair in pairs
    })

    if not pairs:
        raise ValueError(
            "At least one expert pair is required."
        )

    if int(
        grad_accum
    ) <= 0:
        raise ValueError(
            "grad_accum must be positive."
        )

    if int(
        epochs
    ) <= 0:
        raise ValueError(
            "epochs must be positive."
        )

    if int(
        max_updates
    ) <= 0:
        raise ValueError(
            "max_updates must be positive."
        )

    run_seed = (
        1000000
        + int(
            order_seed
        )
        + 10000
        * _capability_number(
            CURRENT_CAPABILITY
        )
    )

    torch.manual_seed(
        run_seed
    )

    torch.cuda.manual_seed_all(
        run_seed
    )

    bank = SurgicalExpertBank(
        pairs
    )

    expected_params = (
        len(
            pairs
        )
        * params_per_expert
    )

    if (
        bank.trainable_parameter_count
        != expected_params
    ):
        raise RuntimeError(
            "Trainable parameter budget mismatch."
        )

    guard = capture_routed_base_guard(
        pairs
    )

    optimizer = None
    checkpointing_enabled = False

    history = []
    raw_step = 0
    update_step = 0
    accum_count = 0

    try:
        bank.install()

        for p in (
            model.parameters()
        ):
            p.requires_grad_(
                False
            )

        _configure_grad_checkpointing()
        checkpointing_enabled = True

        # Training mode is needed by Transformers gradient-checkpointing
        # layers. RNG is explicitly seeded above.
        model.train()
        bank.train()

        optimizer = (
            torch.optim.AdamW(
                bank.parameters(),
                lr=float(
                    lr
                ),
                betas=(
                    0.9,
                    0.95,
                ),
                weight_decay=0.01,
            )
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        initial_state = {
            k: (
                v.detach()
                .cpu()
                .clone()
            )
            for (
                k,
                v,
            ) in (
                bank.state_dict()
                .items()
            )
        }

        rng = (
            np.random.default_rng(
                int(
                    order_seed
                )
            )
        )

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(
            int(
                epochs
            )
        ):
            order = (
                rng.permutation(
                    len(
                        TRAIN_CASES
                    )
                )
                .tolist()
            )

            for (
                position,
                case_idx,
            ) in enumerate(
                order
            ):
                case = TRAIN_CASES[
                    int(
                        case_idx
                    )
                ]

                raw_step += 1
                accum_count += 1

                with torch.autocast(
                    "cuda",
                    dtype=torch.bfloat16,
                ):
                    out = model(
                        input_ids=case[
                            "input_ids"
                        ],
                        attention_mask=case[
                            "attention_mask"
                        ],
                        use_cache=False,
                        logits_to_keep=case[
                            "pred_positions"
                        ],
                        return_dict=True,
                    )

                    logits = (
                        out.logits
                        .float()
                    )

                    loss = (
                        F.cross_entropy(
                            logits.reshape(
                                -1,
                                logits.shape[
                                    -1
                                ],
                            ),
                            case[
                                "targets"
                            ].reshape(
                                -1
                            ),
                        )
                    )

                if not torch.isfinite(
                    loss
                ):
                    raise RuntimeError(
                        f"Non-finite loss in {selector} at raw step {raw_step}."
                    )

                # Accumulate raw per-example gradients, then divide by the
                # ACTUAL window size immediately before the optimizer step.
                loss.backward()

                is_last_example = (
                    epoch
                    == int(
                        epochs
                    )
                    - 1
                    and position
                    == len(
                        order
                    )
                    - 1
                )

                should_step = (
                    accum_count
                    >= int(
                        grad_accum
                    )
                    or is_last_example
                )

                if should_step:
                    if (
                        accum_count
                        <= 0
                    ):
                        raise RuntimeError(
                            "Internal accumulation counter error."
                        )

                    for p in (
                        bank.parameters()
                    ):
                        if (
                            p.grad
                            is not None
                        ):
                            p.grad.div_(
                                float(
                                    accum_count
                                )
                            )

                    grad_norm = (
                        torch.nn.utils.clip_grad_norm_(
                            bank.parameters(),
                            1.0,
                        )
                    )

                    if not bool(
                        torch.isfinite(
                            grad_norm
                        ).item()
                    ):
                        raise RuntimeError(
                            "Non-finite gradient norm during training."
                        )

                    optimizer.step()

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    update_step += 1

                    history.append({
                        "selector": selector,
                        "budget_k": len(
                            pairs
                        ),
                        "order_seed": int(
                            order_seed
                        ),
                        "lr": float(
                            lr
                        ),
                        "update_step": int(
                            update_step
                        ),
                        "raw_step": int(
                            raw_step
                        ),
                        "accumulated_examples": int(
                            accum_count
                        ),
                        "loss": float(
                            loss.detach()
                            .item()
                        ),
                        "grad_norm": float(
                            grad_norm
                        ),
                    })

                    accum_count = 0

                del (
                    out,
                    logits,
                    loss,
                )

                if (
                    update_step
                    >= int(
                        max_updates
                    )
                ):
                    break

            if (
                update_step
                >= int(
                    max_updates
                )
            ):
                break

        if (
            accum_count
            != 0
        ):
            raise RuntimeError(
                "Training ended with unstepped accumulated gradients."
            )

        if (
            update_step
            <= 0
        ):
            raise RuntimeError(
                "Training arm completed zero optimizer updates."
            )

        model.eval()
        bank.eval()

        adapter_metrics = (
            evaluate_batch_against_base(
                eval_batch,
                eval_df,
                eval_base_detail,
            )
        )

        final_state = {
            k: (
                v.detach()
                .cpu()
                .clone()
            )
            for (
                k,
                v,
            ) in (
                bank.state_dict()
                .items()
            )
        }

        delta_l2 = (
            state_delta_l2(
                initial_state,
                final_state,
            )
        )

        base_l2 = (
            state_l2(
                initial_state
            )
        )

        # Return selected layers to Laguna's native expert forward, then
        # physically merge the trained BF16 slices.
        bank.restore()

        try:
            copy_bank_into_base(
                bank,
                pairs,
            )

            model.eval()

            native_metrics = (
                evaluate_batch_against_base(
                    eval_batch,
                    eval_df,
                    eval_base_detail,
                )
            )

            delta_merge_max_nll_diff = float(
                np.max(
                    np.abs(
                        np.asarray(
                            adapter_metrics[
                                "per_example_nll"
                            ]
                        )
                        - np.asarray(
                            native_metrics[
                                "per_example_nll"
                            ]
                        )
                    )
                )
            )

        finally:
            restore_routed_base_guard(
                guard
            )

        assert_routed_base_unchanged(
            guard
        )

        if (
            delta_merge_max_nll_diff
            > DELTA_MERGE_WARN_NLL
        ):
            print(
                "WARNING:",
                selector,
                "adapter/native max ΔNLL =",
                f"{delta_merge_max_nll_diff:.4f}",
                "(diagnostic only)",
            )

        result = {
            "selector": selector,
            "pairs": pairs,
            "budget_k": len(
                pairs
            ),
            "order_seed": int(
                order_seed
            ),
            "lr": float(
                lr
            ),
            "epochs": int(
                epochs
            ),
            "max_updates": int(
                max_updates
            ),
            "updates": int(
                update_step
            ),
            "trainable_params": int(
                bank.trainable_parameter_count
            ),
            "mean_grad_norm": float(
                np.mean([
                    h[
                        "grad_norm"
                    ]
                    for h in (
                        history
                    )
                ])
            ),
            "parameter_delta_l2": float(
                delta_l2
            ),
            "relative_parameter_delta": float(
                delta_l2
                / max(
                    base_l2,
                    1e-12,
                )
            ),
            "peak_gpu_gib": float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
            "delta_merge_max_nll_diff": (
                delta_merge_max_nll_diff
            ),
            **{
                k: v
                for (
                    k,
                    v,
                ) in (
                    adapter_metrics.items()
                )
                if k
                != "per_example_nll"
            },
            "per_example_nll": (
                adapter_metrics[
                    "per_example_nll"
                ]
            ),
        }

        for (
            key,
            value,
        ) in (
            native_metrics.items()
        ):
            if (
                key
                == "per_example_nll"
            ):
                continue

            result[
                f"native_{key}"
            ] = value

        result[
            "native_per_example_nll"
        ] = (
            native_metrics[
                "per_example_nll"
            ]
        )

        result[
            "history"
        ] = history

        return result

    finally:
        # restore() is idempotent and also handles partial installation.
        bank.restore()

        if (
            checkpointing_enabled
        ):
            _disable_grad_checkpointing()

        model.eval()

        # Safe even if native merge/evaluation raised halfway through.
        restore_routed_base_guard(
            guard
        )

        assert_routed_base_unchanged(
            guard
        )

        if (
            optimizer
            is not None
        ):
            del optimizer

        del bank
        del guard

        gc.collect()
        torch.cuda.empty_cache()

## 18 — Build selection, training, and two calibration contexts (no final scoring)

In [ ]:
def _capability_number(capability):
    if str(capability) != "gsm8k":
        raise ValueError(f"Unknown v10 capability: {capability}")
    return 0


selection_df = (
    benchmark_df[benchmark_df["split"] == "selection"]
    .reset_index(drop=True)
)
train_target_df = (
    benchmark_df[benchmark_df["split"] == "train"]
    .reset_index(drop=True)
)
lr_calibration_df = (
    benchmark_df[benchmark_df["split"] == "lr_calibration"]
    .reset_index(drop=True)
)
dose_calibration_df = (
    benchmark_df[benchmark_df["split"] == "dose_calibration"]
    .reset_index(drop=True)
)

if len(selection_df) != SELECTION_TARGET_N + SELECTION_CONTROL_N:
    raise RuntimeError("Selection size mismatch.")
if len(train_target_df) != TRAIN_TARGET_N:
    raise RuntimeError("Training split size mismatch.")
if not bool((train_target_df["kind"] == "target").all()):
    raise RuntimeError("Training split contains non-target rows.")
if len(lr_calibration_df) != LR_CAL_TARGET_N + LR_CAL_CONTROL_N:
    raise RuntimeError("LR calibration size mismatch.")
if len(dose_calibration_df) != DOSE_CAL_TARGET_N + DOSE_CAL_CONTROL_N:
    raise RuntimeError("Dose calibration size mismatch.")

# The final rows exist in the frozen snapshot, but there is intentionally no
# final_test_df/final_generation_df variable before FROZEN_BEFORE_FINAL_TEST.json.

SELECTION_BATCH = build_scoring_batch(selection_df)
assert_teacher_forcing_alignment(
    SELECTION_BATCH,
    label="selection",
)
selection_target_mask = (
    selection_df["kind"].values == "target"
)
selection_control_mask = (
    selection_df["kind"].values == "control"
)
SELECTION_BASE_NLL = score_batch(SELECTION_BATCH)

target_selection_ordered = stable_hash_order(
    selection_df[selection_df.kind == "target"].reset_index(drop=True),
    "example_id",
    "v10_target_grad_probe",
)
control_selection_ordered = stable_hash_order(
    selection_df[selection_df.kind == "control"].reset_index(drop=True),
    "example_id",
    "v10_control_grad_probe",
)

SELECTION_TARGET_CASES = [
    make_training_case(r.prompt, r.reference)
    for r in target_selection_ordered.itertuples(index=False)
]
SELECTION_CONTROL_CASES = [
    make_training_case(r.prompt, r.reference)
    for r in control_selection_ordered.itertuples(index=False)
]
TRAIN_CASES = [
    make_training_case(r.prompt, r.reference)
    for r in train_target_df.itertuples(index=False)
]

_lr_batches = build_scoring_batches(
    lr_calibration_df,
    batch_size=EVAL_BATCH_SIZE,
)
LR_CAL_BASE_DETAIL = score_batches_detailed(_lr_batches)
del _lr_batches

_dose_batches = build_scoring_batches(
    dose_calibration_df,
    batch_size=EVAL_BATCH_SIZE,
)
DOSE_CAL_BASE_DETAIL = score_batches_detailed(_dose_batches)
del _dose_batches

lr_generation_df = (
    lr_calibration_df[lr_calibration_df.kind == "target"]
    .reset_index(drop=True)
)
dose_generation_df = (
    dose_calibration_df[dose_calibration_df.kind == "target"]
    .reset_index(drop=True)
)

print(
    "Selection baseline target/control:",
    float(SELECTION_BASE_NLL[selection_target_mask].mean()),
    float(SELECTION_BASE_NLL[selection_control_mask].mean()),
)
print("Training cases:", len(TRAIN_CASES))
print("LR calibration target rows:", len(lr_generation_df))
print("Dose calibration target rows:", len(dose_generation_df))
print("No final metrics have been accessed.")

## 19 — Scoring and exact expert-surgery preflight

In [ ]:
probe_df = selection_df.head(8).reset_index(drop=True)
probe_batch = build_scoring_batch(probe_df)
assert_teacher_forcing_alignment(
    probe_batch,
    label="preflight",
)

verify_routed_bank_equivalence(
    (25, 168),
    probe_df=probe_df,
    tol=1e-5,
)
verify_native_merge_roundtrip(
    (25, 168),
    probe_df=probe_df,
    tol=1e-5,
)

del probe_batch
gc.collect()
torch.cuda.empty_cache()

print("Teacher forcing / exact-baseline expert surgery: PASS")

## 20 — Global BF16 expert-gradient screen over all 9,984 routed positions

In [ ]:
def _validate_global_pair_table(
    df,
    label,
):
    expected = (
        len(SPARSE_LAYERS)
        * cfg.num_experts
    )

    if len(df) != expected:
        raise RuntimeError(
            f"{label}: row count {len(df)} != {expected}."
        )

    if df[
        [
            "layer",
            "expert",
        ]
    ].duplicated().any():
        raise RuntimeError(
            f"{label}: duplicate (layer, expert) rows."
        )

    expected_layers = set(
        map(
            int,
            SPARSE_LAYERS,
        )
    )

    found_layers = set(
        df[
            "layer"
        ].astype(int)
    )

    if found_layers != expected_layers:
        raise RuntimeError(
            f"{label}: sparse-layer coverage mismatch."
        )

    counts = (
        df.groupby(
            "layer"
        )[
            "expert"
        ]
        .nunique()
    )

    if not bool(
        (
            counts
            == int(
                cfg.num_experts
            )
        ).all()
    ):
        raise RuntimeError(
            f"{label}: one or more layers do not contain "
            f"{cfg.num_experts} unique experts."
        )

    return True



def run_global_gradient_screen():
    path = RESULTS / "global_gradient_screen.csv"

    partial = read_checkpoint_csv(
        path,
        required_columns=[
            "layer",
            "expert",
            "global_target_grad_bf16",
            "global_control_grad_bf16",
            "global_grad_specific_bf16",
            "global_grad_ratio_bf16",
            "scan_peak_gpu_gib",
        ],
        dedupe_keys=[
            "layer",
            "expert",
        ],
    )

    completed_layers = set()

    if not partial.empty:
        counts = partial.groupby(
            "layer"
        ).size()

        completed_layers = {
            int(layer)
            for layer, count in counts.items()
            if int(count) == cfg.num_experts
        }

    print(
        "Global gradient completed layers:",
        len(completed_layers),
        "/",
        len(SPARSE_LAYERS),
    )

    for layer_idx in tqdm(
        SPARSE_LAYERS,
        desc=f"{CURRENT_CAPABILITY}: global gradient",
    ):
        layer_idx = int(layer_idx)

        if layer_idx in completed_layers:
            continue

        torch.cuda.reset_peak_memory_stats()

        target_norm, control_norm = (
            scan_layer_fused_gradients(
                layer_idx,
                SELECTION_TARGET_CASES,
                SELECTION_CONTROL_CASES,
                max_cases=GLOBAL_GRAD_PROBE_CASES,
            )
        )

        layer_rows = []

        for expert_id in range(
            cfg.num_experts
        ):
            gt = float(
                target_norm[
                    expert_id
                ]
            )
            gc_ = float(
                control_norm[
                    expert_id
                ]
            )

            layer_rows.append({
                "layer": layer_idx,
                "expert": int(expert_id),
                "global_target_grad_bf16": gt,
                "global_control_grad_bf16": gc_,
                "global_grad_specific_bf16": (
                    gt
                    - CONTROL_PENALTY
                    * gc_
                ),
                "global_grad_ratio_bf16": (
                    gt
                    / (gc_ + 1e-12)
                ),
                "scan_peak_gpu_gib": float(
                    torch.cuda.max_memory_allocated()
                    / 2**30
                ),
            })

        if partial.empty:
            partial = pd.DataFrame(
                layer_rows
            )
        else:
            partial = pd.concat(
                [
                    partial[
                        partial["layer"]
                        != layer_idx
                    ],
                    pd.DataFrame(
                        layer_rows
                    ),
                ],
                ignore_index=True,
            )

        atomic_to_csv(
            partial,
            path,
            index=False,
        )

    df = pd.read_csv(path)

    _validate_global_pair_table(
        df,
        "global gradient result",
    )

    if not bool(
        np.isfinite(
            df[
                [
                    "global_target_grad_bf16",
                    "global_control_grad_bf16",
                    "global_grad_specific_bf16",
                ]
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ):
        raise RuntimeError(
            "Global gradient screen contains non-finite scores."
        )

    return df

## 21 — Precise FP32 expert geometry and three K=4 expert selectors

In [ ]:
GLOBAL_GRAD_DF = run_global_gradient_screen()

raw_candidates = (
    GLOBAL_GRAD_DF
    .sort_values("global_target_grad_bf16", ascending=False)
    .head(SELECTOR_TOP_N)
)

legacy_specific_candidates = (
    GLOBAL_GRAD_DF
    .sort_values("global_grad_specific_bf16", ascending=False)
    .head(SELECTOR_TOP_N)
)

ratio_candidates = (
    GLOBAL_GRAD_DF
    .sort_values("global_grad_ratio_bf16", ascending=False)
    .head(SELECTOR_TOP_N)
)

required_pairs = {
    (int(r.layer), int(r.expert))
    for frame in [
        raw_candidates,
        legacy_specific_candidates,
        ratio_candidates,
    ]
    for r in frame.itertuples(index=False)
}

PRECISE_PATH = RESULTS / "selection_precise_expert_geometry.csv"

precise_df = read_checkpoint_csv(
    PRECISE_PATH,
    required_columns=[
        "layer",
        "expert",
        "precise_target_grad_l2",
        "precise_control_grad_l2",
        "precise_grad_dot",
        "precise_grad_cosine",
        "precise_grad_specific",
        "precise_grad_ratio",
        "precise_target_orthogonal_l2",
        "precise_target_orthogonal_fraction",
        "precise_abs_grad_cosine",
    ],
    dedupe_keys=["layer", "expert"],
)

done = {
    (int(r.layer), int(r.expert))
    for r in precise_df.itertuples(index=False)
}
rows = precise_df.to_dict(orient="records")

for pair in tqdm(
    sorted(required_pairs),
    desc="precise expert gradient geometry",
):
    if pair in done:
        continue

    metrics = precise_gradient_geometry(
        pair,
        SELECTION_TARGET_CASES,
        SELECTION_CONTROL_CASES,
        max_cases=PRECISE_GRAD_CASES,
    )

    rows.append({
        "layer": int(pair[0]),
        "expert": int(pair[1]),
        **metrics,
    })

    precise_df = pd.DataFrame(rows)
    atomic_to_csv(
        precise_df,
        PRECISE_PATH,
        index=False,
    )
    done.add(pair)

precise_df = (
    pd.read_csv(PRECISE_PATH)
    .drop_duplicates(["layer", "expert"], keep="last")
)

raw_set = {
    (int(r.layer), int(r.expert))
    for r in raw_candidates.itertuples(index=False)
}
legacy_set = {
    (int(r.layer), int(r.expert))
    for r in legacy_specific_candidates.itertuples(index=False)
}

raw_precise = (
    precise_df[
        precise_df.apply(
            lambda r: (
                int(r["layer"]),
                int(r["expert"]),
            ) in raw_set,
            axis=1,
        )
    ]
    .sort_values("precise_target_grad_l2", ascending=False)
)

legacy_precise = (
    precise_df[
        precise_df.apply(
            lambda r: (
                int(r["layer"]),
                int(r["expert"]),
            ) in legacy_set,
            axis=1,
        )
    ]
    .sort_values("precise_grad_specific", ascending=False)
)

contrastive_precise = (
    precise_df
    .sort_values("precise_target_orthogonal_l2", ascending=False)
)

expert_sets = {
    "gradient_experts": [
        (int(r.layer), int(r.expert))
        for r in raw_precise.head(EXPERT_BUDGET_K).itertuples(index=False)
    ],
    "gradient_specific_experts": [
        (int(r.layer), int(r.expert))
        for r in legacy_precise.head(EXPERT_BUDGET_K).itertuples(index=False)
    ],
    "contrastive_experts": [
        (int(r.layer), int(r.expert))
        for r in contrastive_precise.head(EXPERT_BUDGET_K).itertuples(index=False)
    ],
}

for method, pairs in expert_sets.items():
    if len(pairs) != EXPERT_BUDGET_K:
        raise RuntimeError(f"{method}: wrong K.")
    if len(set(pairs)) != EXPERT_BUDGET_K:
        raise RuntimeError(f"{method}: duplicate selected experts.")

    for pair in pairs:
        verify_routed_bank_equivalence(
            pair,
            tol=1e-5,
        )

    verify_native_merge_roundtrip_pairs(
        pairs,
        tol=1e-5,
    )

EXPERT_SELECTOR_PATH = RESULTS / "frozen_expert_selectors.json"

atomic_write_text(
    EXPERT_SELECTOR_PATH,
    json.dumps(
        {
            k: [list(map(int, p)) for p in v]
            for k, v in expert_sets.items()
        },
        indent=2,
    ),
)

print("Frozen selection-only expert sets:")
for method, pairs in expert_sets.items():
    print(" ", method, pairs)

display(
    contrastive_precise[
        [
            "layer",
            "expert",
            "precise_target_grad_l2",
            "precise_control_grad_l2",
            "precise_grad_cosine",
            "precise_target_orthogonal_l2",
            "precise_target_orthogonal_fraction",
        ]
    ].head(12)
)

## 22 — Attention target/control gradient geometry for contrastive LoRA placement

In [ ]:
def _unwrap_linear(module):
    current = module
    seen = set()

    while not isinstance(current, nn.Linear):
        obj_id = id(current)
        if obj_id in seen:
            raise RuntimeError("Cycle while unwrapping PEFT linear.")
        seen.add(obj_id)

        if not hasattr(current, "base_layer"):
            raise RuntimeError(
                f"Expected nn.Linear or PEFT wrapper, got {type(current)}."
            )

        current = current.base_layer

    return current


def _get_attention_lora_base_modules(layer_idx):
    attn = model.model.layers[int(layer_idx)].self_attn
    modules = []

    for short_name in LORA_TARGET_MODULES:
        module = getattr(attn, short_name, None)

        if module is None:
            raise RuntimeError(
                f"Layer {layer_idx} missing {short_name}."
            )

        modules.append(
            (
                short_name,
                _unwrap_linear(module),
            )
        )

    return modules


def _backprop_attention_cases(
    layer_idx,
    cases,
    max_cases,
):
    modules = _get_attention_lora_base_modules(layer_idx)
    weights = [module.weight for _, module in modules]

    for p in model.parameters():
        p.requires_grad_(False)

    for p in weights:
        p.requires_grad_(True)
        p.grad = None

    model.eval()

    try:
        _gradient_objective_for_cases(
            cases,
            max_cases,
        )

        grads = []

        for p in weights:
            if p.grad is None:
                grads.append(
                    torch.zeros_like(
                        p,
                        device="cpu",
                        dtype=torch.float32,
                    )
                )
            else:
                g = (
                    p.grad
                    .detach()
                    .float()
                    .cpu()
                    .clone()
                )

                if not bool(torch.isfinite(g).all()):
                    raise RuntimeError(
                        f"Non-finite attention gradient at layer {layer_idx}."
                    )

                grads.append(g)

        return grads

    finally:
        for p in weights:
            p.grad = None
            p.requires_grad_(False)

        gc.collect()
        torch.cuda.empty_cache()


def attention_gradient_geometry(layer_idx):
    target_grads = _backprop_attention_cases(
        layer_idx,
        SELECTION_TARGET_CASES,
        ATTENTION_GRAD_PROBE_CASES,
    )

    control_grads = _backprop_attention_cases(
        layer_idx,
        SELECTION_CONTROL_CASES,
        ATTENTION_GRAD_PROBE_CASES,
    )

    target_sq = 0.0
    control_sq = 0.0
    dot = 0.0

    for gt, gc_ in zip(
        target_grads,
        control_grads,
    ):
        target_sq += float(
            torch.sum(gt * gt).item()
        )
        control_sq += float(
            torch.sum(gc_ * gc_).item()
        )
        dot += float(
            torch.sum(gt * gc_).item()
        )

    target_l2 = float(np.sqrt(target_sq))
    control_l2 = float(np.sqrt(control_sq))

    cosine = float(
        dot
        / max(
            target_l2 * control_l2,
            1e-12,
        )
    )

    if control_sq <= 1e-24:
        orth_sq = target_sq
    else:
        orth_sq = max(
            target_sq
            - (dot * dot) / control_sq,
            0.0,
        )

    orth_l2 = float(np.sqrt(orth_sq))

    del target_grads
    del control_grads
    gc.collect()

    return {
        "layer": int(layer_idx),
        "attention_target_grad_l2": target_l2,
        "attention_control_grad_l2": control_l2,
        "attention_grad_dot": float(dot),
        "attention_grad_cosine": cosine,
        "attention_abs_grad_cosine": float(abs(cosine)),
        "attention_target_orthogonal_l2": orth_l2,
        "attention_target_orthogonal_fraction": float(
            orth_l2 / max(target_l2, 1e-12)
        ),
        "attention_grad_ratio": float(
            target_l2 / (control_l2 + 1e-12)
        ),
    }


ATTENTION_GEOMETRY_PATH = (
    RESULTS / "attention_gradient_geometry.csv"
)

attention_df = read_checkpoint_csv(
    ATTENTION_GEOMETRY_PATH,
    required_columns=[
        "layer",
        "attention_target_grad_l2",
        "attention_control_grad_l2",
        "attention_grad_dot",
        "attention_grad_cosine",
        "attention_abs_grad_cosine",
        "attention_target_orthogonal_l2",
        "attention_target_orthogonal_fraction",
        "attention_grad_ratio",
    ],
    dedupe_keys=["layer"],
)

done_layers = (
    set(attention_df["layer"].astype(int).tolist())
    if not attention_df.empty
    else set()
)

rows = attention_df.to_dict(orient="records")

for layer_idx in tqdm(
    range(cfg.num_hidden_layers),
    desc="attention contrastive gradient geometry",
):
    layer_idx = int(layer_idx)

    if layer_idx in done_layers:
        continue

    rows.append(
        attention_gradient_geometry(layer_idx)
    )

    attention_df = pd.DataFrame(rows)

    atomic_to_csv(
        attention_df,
        ATTENTION_GEOMETRY_PATH,
        index=False,
    )

    done_layers.add(layer_idx)

attention_df = (
    pd.read_csv(ATTENTION_GEOMETRY_PATH)
    .drop_duplicates("layer", keep="last")
)

if (
    len(attention_df) != cfg.num_hidden_layers
    or set(attention_df["layer"].astype(int))
    != set(range(cfg.num_hidden_layers))
):
    raise RuntimeError(
        "Attention geometry coverage mismatch."
    )

if not bool(
    np.isfinite(
        attention_df[
            [
                "attention_target_grad_l2",
                "attention_control_grad_l2",
                "attention_grad_cosine",
                "attention_target_orthogonal_l2",
            ]
        ].to_numpy(dtype=np.float64)
    ).all()
):
    raise RuntimeError(
        "Non-finite attention gradient geometry."
    )

attention_df = (
    attention_df
    .sort_values(
        "attention_target_orthogonal_l2",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(attention_df.head(12))

## 23 — Match LoRA parameter budgets and freeze attention locations

In [ ]:
def collect_attention_lora_modules(layers):
    wanted_layers = sorted({int(x) for x in layers})

    if not wanted_layers:
        raise RuntimeError("LoRA layer set is empty.")

    invalid = [
        x
        for x in wanted_layers
        if not (
            0
            <= x
            < int(cfg.num_hidden_layers)
        )
    ]

    if invalid:
        raise RuntimeError(
            f"Invalid LoRA layer indices: {invalid}"
        )

    rows = []

    for layer_idx in wanted_layers:
        for (
            short_name,
            module,
        ) in _get_attention_lora_base_modules(
            layer_idx
        ):
            rows.append({
                "layer": int(layer_idx),
                "short_name": str(short_name),
                "in_features": int(
                    module.in_features
                ),
                "out_features": int(
                    module.out_features
                ),
                "rank_coefficient": int(
                    module.in_features
                    + module.out_features
                ),
            })

    modules = pd.DataFrame(rows)

    expected_rows = (
        len(wanted_layers)
        * len(LORA_TARGET_MODULES)
    )

    if len(modules) != expected_rows:
        raise RuntimeError(
            "Attention-LoRA module discovery count mismatch: "
            f"{len(modules)} != {expected_rows}"
        )

    return modules


def choose_lora_rank(
    layers,
    target_params,
):
    modules = (
        collect_attention_lora_modules(
            layers
        )
    )

    coefficient = int(
        modules[
            "rank_coefficient"
        ].sum()
    )

    ideal = float(
        target_params
    ) / float(
        coefficient
    )

    candidates = sorted({
        max(
            1,
            int(
                math.floor(
                    ideal
                )
            ),
        ),
        max(
            1,
            int(
                round(
                    ideal
                )
            ),
        ),
        max(
            1,
            int(
                math.ceil(
                    ideal
                )
            ),
        ),
    })

    rank = min(
        candidates,
        key=lambda r: abs(
            r * coefficient
            - target_params
        ),
    )

    predicted = int(
        rank
        * coefficient
    )

    rel_error = abs(
        predicted
        - target_params
    ) / float(
        target_params
    )

    return {
        "rank": int(
            rank
        ),
        "predicted_trainable_params": int(
            predicted
        ),
        "budget_rel_error": float(
            rel_error
        ),
        "coefficient": int(
            coefficient
        ),
        "module_count": int(
            len(
                modules
            )
        ),
        "layers": [
            int(x)
            for x in layers
        ],
    }


EXPERT_TARGET_PARAMS = int(
    EXPERT_BUDGET_K
    * params_per_expert
)

contrastive_lora_layers = (
    attention_df
    .sort_values(
        "attention_target_orthogonal_l2",
        ascending=False,
    )
    .head(
        GUIDED_LORA_N_LAYERS
    )[
        "layer"
    ]
    .astype(int)
    .tolist()
)

if (
    len(
        contrastive_lora_layers
    )
    != GUIDED_LORA_N_LAYERS
):
    raise RuntimeError(
        "Could not construct contrastive LoRA layers."
    )

rng = np.random.default_rng(
    RANDOM_LAYER_SEED
)

random_lora_layers = sorted(
    map(
        int,
        rng.choice(
            np.arange(
                cfg.num_hidden_layers,
                dtype=np.int64,
            ),
            size=(
                RANDOM_LORA_N_LAYERS
            ),
            replace=False,
        ),
    )
)

standard_lora_layers = list(
    range(
        cfg.num_hidden_layers
    )
)

lora_layer_sets = {
    "standard_lora": (
        standard_lora_layers
    ),
    "random_layer_lora": (
        random_lora_layers
    ),
    "contrastive_lora": (
        contrastive_lora_layers
    ),
}

lora_plans = {}

for method, layers in (
    lora_layer_sets.items()
):
    plan = choose_lora_rank(
        layers,
        EXPERT_TARGET_PARAMS,
    )

    plan[
        "method"
    ] = method

    if (
        plan[
            "budget_rel_error"
        ]
        > MAX_LORA_BUDGET_REL_ERROR
    ):
        raise RuntimeError(
            f"{method}: cannot match expert budget within "
            f"{MAX_LORA_BUDGET_REL_ERROR:.1%}; "
            f"error={plan['budget_rel_error']:.2%}"
        )

    lora_plans[
        method
    ] = plan

LORA_PLAN_PATH = (
    RESULTS
    / "lora_budget_plan.json"
)

atomic_write_text(
    LORA_PLAN_PATH,
    json.dumps(
        {
            "expert_target_params": (
                EXPERT_TARGET_PARAMS
            ),
            "plans": (
                lora_plans
            ),
        },
        indent=2,
    ),
)

print(
    f"K={EXPERT_BUDGET_K} expert budget:",
    f"{EXPERT_TARGET_PARAMS:,}",
)

for method, plan in (
    lora_plans.items()
):
    print(
        method,
        "layers=",
        plan[
            "layers"
        ],
        "rank=",
        plan[
            "rank"
        ],
        "pred=",
        f"{plan['predicted_trainable_params']:,}",
        "error=",
        f"{plan['budget_rel_error']:.3%}",
    )

## 24 — Runtime LoRA injection/no-op/layout/budget preflight

In [ ]:
def _lora_config_from_plan(plan):
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=int(plan["rank"]),
        lora_alpha=int(round(plan["rank"] * LORA_ALPHA_MULTIPLIER)),
        lora_dropout=0.0,
        bias="none",
        target_modules=list(LORA_TARGET_MODULES),
        layers_to_transform=[int(x) for x in plan["layers"]],
        layers_pattern="layers",
        init_lora_weights=True,
    )


def _trainable_parameters():
    return [p for p in model.parameters() if p.requires_grad]


def _delete_adapter_and_freeze(adapter_name):
    if (
        hasattr(model, "peft_config")
        and adapter_name
        in getattr(model, "peft_config", {})
    ):
        model.delete_adapter(
            adapter_name
        )

    for p in model.parameters():
        p.requires_grad_(False)

    model.eval()
    gc.collect()
    torch.cuda.empty_cache()


def _audit_active_lora_layout(plan):
    """Verify exact LoRA layers, modules, and A/B tensor count."""
    trainable_named = [
        (name, p)
        for name, p in model.named_parameters()
        if p.requires_grad
    ]

    if not trainable_named:
        raise RuntimeError(
            "Active LoRA adapter has no trainable parameters."
        )

    bad = [
        name
        for name, _ in trainable_named
        if "lora_" not in name
    ]

    if bad:
        raise RuntimeError(
            "Non-LoRA trainables detected: "
            + repr(bad[:8])
        )

    expected_layers = {
        int(x)
        for x in plan["layers"]
    }

    found_layers = set()

    for name, _ in trainable_named:
        match = re.search(
            r"(?:^|\.)layers\.(\d+)\.",
            name,
        )

        if match is None:
            raise RuntimeError(
                "Cannot recover layer index from LoRA parameter: "
                + name
            )

        found_layers.add(
            int(match.group(1))
        )

        if not any(
            f".{short_name}." in name
            for short_name in LORA_TARGET_MODULES
        ):
            raise RuntimeError(
                "Unexpected LoRA target module: "
                + name
            )

    if found_layers != expected_layers:
        raise RuntimeError(
            "LoRA adapted wrong layers: "
            f"found={sorted(found_layers)} "
            f"expected={sorted(expected_layers)}"
        )

    expected_tensors = (
        len(expected_layers)
        * len(LORA_TARGET_MODULES)
        * 2
    )

    if len(trainable_named) != expected_tensors:
        raise RuntimeError(
            "LoRA trainable tensor count mismatch: "
            f"{len(trainable_named)} != {expected_tensors}"
        )

    return trainable_named


def lora_preflight(method, plan):
    adapter_name = "preflight_" + method + "_" + uuid.uuid4().hex[:8]
    probe_df = selection_df.head(4).reset_index(drop=True)
    probe_batch = build_scoring_batch(probe_df)
    before = score_batch(probe_batch)
    actual_params = None
    no_op_diff = None
    try:
        model.add_adapter(_lora_config_from_plan(plan), adapter_name=adapter_name)
        model.set_adapter(adapter_name)
        if hasattr(model, "enable_adapters"):
            model.enable_adapters()
        trainable_named = _audit_active_lora_layout(plan)
        trainable = [p for _, p in trainable_named]
        actual_params = int(sum(p.numel() for p in trainable))
        if actual_params <= 0:
            raise RuntimeError(f"{method}: adapter has zero trainable parameters.")
        after = score_batch(probe_batch)
        no_op_diff = float(np.max(np.abs(before - after)))
        if no_op_diff > 1e-5:
            raise RuntimeError(
                f"{method}: zero-initialized LoRA changed base scoring; max ΔNLL={no_op_diff}"
            )
    finally:
        _delete_adapter_and_freeze(adapter_name)

    restored = score_batch(probe_batch)
    restore_diff = float(np.max(np.abs(before - restored)))
    del probe_batch
    if restore_diff > 1e-5:
        raise RuntimeError(
            f"{method}: adapter deletion did not restore base scoring; max ΔNLL={restore_diff}"
        )
    rel_error = abs(actual_params - EXPERT_TARGET_PARAMS) / float(EXPERT_TARGET_PARAMS)
    if rel_error > MAX_LORA_BUDGET_REL_ERROR:
        raise RuntimeError(
            f"{method}: actual PEFT budget differs from K={EXPERT_BUDGET_K} experts by {rel_error:.2%}."
        )
    return {
        "actual_trainable_params": int(actual_params),
        "actual_budget_rel_error": float(rel_error),
        "no_op_max_nll_diff": float(no_op_diff),
        "delete_restore_max_nll_diff": float(restore_diff),
    }

for method in list(lora_plans):
    audit = lora_preflight(method, lora_plans[method])
    lora_plans[method].update(audit)
    print(method, audit)

atomic_write_text(
    LORA_PLAN_PATH,
    json.dumps({"expert_target_params": EXPERT_TARGET_PARAMS, "plans": lora_plans}, indent=2),
)
print("LoRA runtime budget/no-op preflight: PASS")

## 25 — Deterministic GSM8K final-answer evaluator

In [ ]:
_NUMBER_RE = re.compile(r"[+-]?(?:\d[\d,]*)(?:\.\d+)?(?:/\d+)?%?")


def canonical_number_string(value):
    if value is None:
        return None
    s = str(value).strip().replace("$", "").replace(",", "").strip()
    if not s:
        return None
    percent = s.endswith("%")
    if percent:
        s = s[:-1].strip()
    try:
        if "/" in s:
            f = Fraction(s)
            normalized = f"{f.numerator}/{f.denominator}"
        else:
            d = Decimal(s)
            if d == d.to_integral():
                normalized = str(d.quantize(Decimal(1)))
            else:
                normalized = format(d.normalize(), "f").rstrip("0").rstrip(".")
        return normalized + ("%" if percent else "")
    except (InvalidOperation, ValueError, ZeroDivisionError):
        return s


def extract_final_numeric_answer(text):
    text = str(text)
    marked = re.findall(r"####\s*([^\n\r]+)", text)
    if marked:
        vals = _NUMBER_RE.findall(marked[-1])
        if vals:
            return canonical_number_string(vals[-1])
    vals = _NUMBER_RE.findall(text)
    return None if not vals else canonical_number_string(vals[-1])


@torch.inference_mode()
def evaluate_gsm8k_generation(df, batch_size=None, max_new_tokens=None):
    batch_size = GENERATION_BATCH_SIZE if batch_size is None else int(batch_size)
    max_new_tokens = GENERATION_MAX_NEW_TOKENS if max_new_tokens is None else int(max_new_tokens)
    df = df.reset_index(drop=True).copy()
    if not bool((df["kind"] == "target").all()):
        raise RuntimeError("Generation evaluator accepts target GSM8K rows only.")

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    rows = []
    try:
        for start in tqdm(range(0, len(df), batch_size), desc="GSM8K generation", leave=False):
            part = df.iloc[start:start+batch_size]
            prefixes = [chat_prefix_text(r.prompt) + "\n" for r in part.itertuples(index=False)]
            enc = tokenizer(
                prefixes,
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            )
            enc = {k: v.to("cuda:0") for k, v in enc.items()}
            generated = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            new_tokens = generated[:, enc["input_ids"].shape[1]:]
            texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
            eos_ids = tokenizer.eos_token_id

            if eos_ids is None:
                raise RuntimeError(
                    "Tokenizer has no eos_token_id."
                )

            eos_ids = (
                {int(eos_ids)}
                if isinstance(eos_ids, int)
                else {int(x) for x in eos_ids}
            )
            for row_pos, (row, output_text) in enumerate(zip(part.itertuples(index=False), texts)):
                token_row = new_tokens[row_pos].detach().cpu().tolist()
                hit_cap = not any(int(tok) in eos_ids for tok in token_row)
                pred = extract_final_numeric_answer(output_text)
                gold = canonical_number_string(row.gold_answer)
                rows.append({
                    "example_id": row.example_id,
                    "gold": gold,
                    "pred": pred,
                    "correct": int(pred == gold),
                    "hit_max_new_tokens": int(hit_cap),
                    "generated_text": output_text,
                })
            del generated, new_tokens, enc
            gc.collect(); torch.cuda.empty_cache()
    finally:
        tokenizer.padding_side = old_padding_side

    detail = pd.DataFrame(rows)
    return {
        "accuracy": float(detail["correct"].mean()),
        "n": int(len(detail)),
        "detail": detail,
    }

# Pure parser self-tests before any expensive generation.
_parser_tests = {
    "work\n#### 1,234": "1234",
    "work\n#### -7": "-7",
    "work\n#### 3/4": "3/4",
    "reason 2 then 9": "9",
}
for text, expected in _parser_tests.items():
    got = extract_final_numeric_answer(text)
    if got != expected:
        raise RuntimeError(f"GSM8K answer parser self-test failed: {text!r}: {got!r} != {expected!r}")
print("GSM8K answer parser self-tests: PASS")

## 26 — Snapshot optimizer and behavior-first selection rule

In [ ]:
def _seed_training_run(order_seed):
    run_seed = 1_700_000 + int(order_seed)
    torch.manual_seed(run_seed)
    torch.cuda.manual_seed_all(run_seed)
    np.random.seed(run_seed % (2**32 - 1))
    return run_seed


def _snapshot_named_parameters(trainable_named):
    return {
        name: p.detach().cpu().clone()
        for name, p in trainable_named
    }


def _load_named_snapshot(trainable_named, snapshot):
    current = {
        name: p
        for name, p in trainable_named
    }

    if set(current) != set(snapshot):
        raise RuntimeError(
            "Snapshot/trainable parameter name mismatch."
        )

    with torch.no_grad():
        for name, p in current.items():
            p.copy_(
                snapshot[name].to(
                    device=p.device,
                    dtype=p.dtype,
                )
            )


def _run_optimizer_with_snapshots(
    trainable_named,
    order_seed,
    lr,
    checkpoint_steps,
):
    trainable_named = list(trainable_named)
    trainable_params = [
        p
        for _, p in trainable_named
    ]

    if not trainable_params:
        raise RuntimeError(
            "No trainable parameters supplied."
        )

    requested = sorted({
        int(x)
        for x in checkpoint_steps
    })

    if not requested:
        raise RuntimeError(
            "No checkpoint steps requested."
        )

    if min(requested) < 0:
        raise RuntimeError(
            "Negative checkpoint step."
        )

    max_updates = max(requested)

    snapshots = {
        0: _snapshot_named_parameters(
            trainable_named
        )
    }

    if max_updates == 0:
        return [], snapshots

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=float(lr),
        betas=tuple(
            OPTIMIZER_BETAS
        ),
        weight_decay=float(
            OPTIMIZER_WEIGHT_DECAY
        ),
    )

    rng = np.random.default_rng(
        int(order_seed)
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    history = []
    raw_step = 0
    update_step = 0
    accum_count = 0

    try:
        for epoch in range(
            int(TRAIN_EPOCHS)
        ):
            order = (
                rng.permutation(
                    len(TRAIN_CASES)
                )
                .tolist()
            )

            for position, case_idx in enumerate(
                order
            ):
                case = TRAIN_CASES[
                    int(case_idx)
                ]

                raw_step += 1
                accum_count += 1

                with torch.autocast(
                    "cuda",
                    dtype=torch.bfloat16,
                ):
                    out = model(
                        input_ids=case[
                            "input_ids"
                        ],
                        attention_mask=case[
                            "attention_mask"
                        ],
                        use_cache=False,
                        logits_to_keep=case[
                            "pred_positions"
                        ],
                        return_dict=True,
                    )

                    logits = out.logits.float()

                    loss = F.cross_entropy(
                        logits.reshape(
                            -1,
                            logits.shape[
                                -1
                            ],
                        ),
                        case[
                            "targets"
                        ].reshape(
                            -1
                        ),
                    )

                if not bool(
                    torch.isfinite(
                        loss
                    ).item()
                ):
                    raise RuntimeError(
                        f"Non-finite training loss at raw step {raw_step}."
                    )

                loss.backward()

                is_last = (
                    epoch
                    == int(TRAIN_EPOCHS)
                    - 1
                    and position
                    == len(order)
                    - 1
                )

                should_step = (
                    accum_count
                    >= int(
                        TRAIN_GRAD_ACCUM
                    )
                    or is_last
                )

                if should_step:
                    for p in trainable_params:
                        if p.grad is not None:
                            p.grad.div_(
                                float(
                                    accum_count
                                )
                            )

                    grad_norm = (
                        torch.nn.utils.clip_grad_norm_(
                            trainable_params,
                            float(
                                GRAD_CLIP_NORM
                            ),
                        )
                    )

                    if not bool(
                        torch.isfinite(
                            grad_norm
                        ).item()
                    ):
                        raise RuntimeError(
                            "Non-finite gradient norm."
                        )

                    optimizer.step()
                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    update_step += 1

                    history.append({
                        "epoch": int(epoch),
                        "update_step": int(
                            update_step
                        ),
                        "raw_step": int(
                            raw_step
                        ),
                        "accumulated_examples": int(
                            accum_count
                        ),
                        "loss": float(
                            loss.detach().item()
                        ),
                        "grad_norm": float(
                            grad_norm
                        ),
                    })

                    accum_count = 0

                    if update_step in requested:
                        snapshots[
                            int(update_step)
                        ] = (
                            _snapshot_named_parameters(
                                trainable_named
                            )
                        )

                del out
                del logits
                del loss

                if (
                    update_step
                    >= max_updates
                ):
                    break

            if (
                update_step
                >= max_updates
            ):
                break

        if accum_count != 0:
            raise RuntimeError(
                "Training ended with unstepped accumulated gradients."
            )

        missing = (
            set(requested)
            - set(snapshots)
        )

        if missing:
            raise RuntimeError(
                "Training did not reach requested checkpoints: "
                + repr(
                    sorted(missing)
                )
            )

        return history, snapshots

    finally:
        del optimizer


def _base_curve_row(
    method,
    family,
    lr,
    eval_df,
    base_detail,
    base_generation_accuracy,
    trainable_params,
):
    target_mask = (
        eval_df["kind"].values
        == "target"
    )
    control_mask = (
        eval_df["kind"].values
        == "control"
    )

    return {
        "method": str(method),
        "family": str(family),
        "lr": float(lr),
        "updates": 0,
        "trainable_params": int(
            trainable_params
        ),
        "target_improvement": 0.0,
        "control_improvement": 0.0,
        "control_abs_shift": 0.0,
        "specific_gain": 0.0,
        "selective_target_gain": 0.0,
        "relative_target_improvement": 0.0,
        "relative_control_improvement": 0.0,
        "generation_accuracy": float(
            base_generation_accuracy
        ),
        "target_nll": float(
            np.asarray(
                base_detail[
                    "nll"
                ]
            )[
                target_mask
            ].mean()
        ),
        "control_nll": float(
            np.asarray(
                base_detail[
                    "nll"
                ]
            )[
                control_mask
            ].mean()
        ),
    }


def choose_behavior_row(
    curve_df,
    allow_zero,
):
    work = curve_df.copy()

    if not allow_zero:
        work = work[
            work[
                "updates"
            ].astype(int)
            > 0
        ]

    if work.empty:
        raise RuntimeError(
            "No candidate behavior rows."
        )

    required = [
        "generation_accuracy",
        "control_abs_shift",
        "target_improvement",
        "updates",
        "lr",
    ]

    if not bool(
        np.isfinite(
            work[
                required
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ):
        raise RuntimeError(
            "Non-finite behavior-selection metric."
        )

    # Lexicographic rule, fixed before final:
    # 1) autonomous GSM8K exact accuracy,
    # 2) smallest magnitude of unrelated control movement,
    # 3) target teacher-forced improvement,
    # 4) fewer updates,
    # 5) smaller LR.
    ranked = (
        work.sort_values(
            [
                "generation_accuracy",
                "control_abs_shift",
                "target_improvement",
                "updates",
                "lr",
            ],
            ascending=[
                False,
                True,
                False,
                True,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    return ranked.iloc[
        0
    ].copy()

## 27 — Native-BF16 expert checkpoint curves

In [ ]:
def _snapshot_delta_l2(
    snapshot0,
    snapshot1,
):
    total = 0.0

    if set(snapshot0) != set(snapshot1):
        raise RuntimeError(
            "Snapshot key mismatch."
        )

    for key in snapshot0:
        a = snapshot0[
            key
        ].float()
        b = snapshot1[
            key
        ].float()
        d = b - a
        total += float(
            torch.sum(
                d * d
            ).item()
        )

    return float(
        np.sqrt(
            total
        )
    )


def run_expert_checkpoint_curve(
    method,
    pairs,
    order_seed,
    lr,
    checkpoint_steps,
    eval_df,
    eval_base_detail,
    generation_df,
    base_generation_accuracy,
    base_generation_detail=None,
    return_generation_details=False,
):
    pairs = sorted({
        tuple(
            map(
                int,
                p,
            )
        )
        for p in pairs
    })

    if len(
        pairs
    ) != EXPERT_BUDGET_K:
        raise RuntimeError(
            f"{method}: expected K={EXPERT_BUDGET_K}, got {len(pairs)}."
        )

    requested = sorted({
        int(x)
        for x in checkpoint_steps
    })

    if 0 not in requested:
        requested = [
            0,
            *requested,
        ]

    _seed_training_run(
        order_seed
    )

    guard = capture_routed_base_guard(
        pairs
    )

    bank = SurgicalExpertBank(
        pairs
    )

    expected_params = (
        len(pairs)
        * params_per_expert
    )

    if (
        bank.trainable_parameter_count
        != expected_params
    ):
        raise RuntimeError(
            f"{method}: expert parameter budget mismatch."
        )

    checkpointing_enabled = False
    history = []
    snapshots = None
    rows = []
    details = {}

    try:
        bank.install()

        for p in model.parameters():
            p.requires_grad_(
                False
            )

        for p in bank.parameters():
            p.requires_grad_(
                True
            )

        _configure_grad_checkpointing()
        checkpointing_enabled = True

        model.train()
        bank.train()

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        t0 = time.time()

        history, snapshots = (
            _run_optimizer_with_snapshots(
                list(
                    bank.named_parameters()
                ),
                order_seed=(
                    order_seed
                ),
                lr=lr,
                checkpoint_steps=(
                    requested
                ),
            )
        )

        train_wall_seconds = float(
            time.time()
            - t0
        )

        peak_gpu_gib = float(
            torch.cuda.max_memory_allocated()
            / 2**30
        )

        # Calibration and final expert metrics use the same deployment path:
        # native BF16 Laguna expert slices, not the delta surrogate.
        bank.restore()

        rows.append(
            _base_curve_row(
                method=method,
                family="full_expert",
                lr=lr,
                eval_df=eval_df,
                base_detail=eval_base_detail,
                base_generation_accuracy=(
                    base_generation_accuracy
                ),
                trainable_params=(
                    bank.trainable_parameter_count
                ),
            )
        )

        if (
            return_generation_details
            and base_generation_detail
            is not None
        ):
            details[
                0
            ] = (
                base_generation_detail
                .copy()
            )

        for step in requested:
            step = int(step)

            if step == 0:
                continue

            _load_named_snapshot(
                list(
                    bank.named_parameters()
                ),
                snapshots[
                    step
                ],
            )

            try:
                copy_bank_into_base(
                    bank,
                    pairs,
                )

                model.eval()

                metrics = (
                    evaluate_df_against_base(
                        eval_df,
                        eval_base_detail,
                        batch_size=(
                            EVAL_BATCH_SIZE
                        ),
                    )
                )

                generation = (
                    evaluate_gsm8k_generation(
                        generation_df
                    )
                )

            finally:
                restore_routed_base_guard(
                    guard
                )

            assert_routed_base_unchanged(
                guard
            )

            row = {
                "method": str(
                    method
                ),
                "family": "full_expert",
                "lr": float(
                    lr
                ),
                "updates": int(
                    step
                ),
                "trainable_params": int(
                    bank.trainable_parameter_count
                ),
                "target_improvement": float(
                    metrics[
                        "target_improvement"
                    ]
                ),
                "control_improvement": float(
                    metrics[
                        "control_improvement"
                    ]
                ),
                "control_abs_shift": float(
                    metrics[
                        "control_abs_shift"
                    ]
                ),
                "specific_gain": float(
                    metrics[
                        "specific_gain"
                    ]
                ),
                "selective_target_gain": float(
                    metrics[
                        "selective_target_gain"
                    ]
                ),
                "relative_target_improvement": float(
                    metrics[
                        "relative_target_improvement"
                    ]
                ),
                "relative_control_improvement": float(
                    metrics[
                        "relative_control_improvement"
                    ]
                ),
                "generation_accuracy": float(
                    generation[
                        "accuracy"
                    ]
                ),
                "target_nll": float(
                    metrics[
                        "target_nll"
                    ]
                ),
                "control_nll": float(
                    metrics[
                        "control_nll"
                    ]
                ),
                "parameter_delta_l2": float(
                    _snapshot_delta_l2(
                        snapshots[
                            0
                        ],
                        snapshots[
                            step
                        ],
                    )
                ),
                "peak_gpu_gib": (
                    peak_gpu_gib
                ),
                "train_wall_seconds": (
                    train_wall_seconds
                ),
            }

            rows.append(
                row
            )

            if return_generation_details:
                details[
                    step
                ] = (
                    generation[
                        "detail"
                    ]
                )

        curve = (
            pd.DataFrame(
                rows
            )
            .sort_values(
                "updates"
            )
            .reset_index(
                drop=True
            )
        )

        return (
            curve,
            details,
            history,
        )

    finally:
        bank.restore()

        if checkpointing_enabled:
            _disable_grad_checkpointing()

        restore_routed_base_guard(
            guard
        )

        assert_routed_base_unchanged(
            guard
        )

        for p in model.parameters():
            p.requires_grad_(
                False
            )

        model.eval()

        del bank
        del guard

        gc.collect()
        torch.cuda.empty_cache()

## 28 — LoRA checkpoint curves with exact adapter restoration

In [ ]:
def run_lora_checkpoint_curve(
    method,
    plan,
    order_seed,
    lr,
    checkpoint_steps,
    eval_df,
    eval_base_detail,
    generation_df,
    base_generation_accuracy,
    base_generation_detail=None,
    return_generation_details=False,
):
    requested = sorted({
        int(x)
        for x in checkpoint_steps
    })

    if 0 not in requested:
        requested = [
            0,
            *requested,
        ]

    adapter_name = (
        method
        + "_"
        + str(
            int(
                order_seed
            )
        )
        + "_"
        + uuid.uuid4().hex[
            :8
        ]
    )

    _seed_training_run(
        order_seed
    )

    probe_df = (
        selection_df
        .head(4)
        .reset_index(
            drop=True
        )
    )

    probe_batch = (
        build_scoring_batch(
            probe_df
        )
    )

    base_probe = score_batch(
        probe_batch
    )

    checkpointing_enabled = False
    history = []
    snapshots = None
    rows = []
    details = {}

    try:
        for p in model.parameters():
            p.requires_grad_(
                False
            )

        model.add_adapter(
            _lora_config_from_plan(
                plan
            ),
            adapter_name=(
                adapter_name
            ),
        )

        model.set_adapter(
            adapter_name
        )

        if hasattr(
            model,
            "enable_adapters"
        ):
            model.enable_adapters()

        trainable_named = (
            _audit_active_lora_layout(
                plan
            )
        )

        trainable_params = int(
            sum(
                p.numel()
                for _, p in (
                    trainable_named
                )
            )
        )

        expected_actual = int(
            plan[
                "actual_trainable_params"
            ]
        )

        if (
            trainable_params
            != expected_actual
        ):
            raise RuntimeError(
                f"{method}: actual LoRA budget changed: "
                f"{trainable_params} != {expected_actual}"
            )

        no_op = score_batch(
            probe_batch
        )

        no_op_diff = float(
            np.max(
                np.abs(
                    base_probe
                    - no_op
                )
            )
        )

        if no_op_diff > 1e-5:
            raise RuntimeError(
                f"{method}: LoRA initialization is not a no-op; "
                f"max ΔNLL={no_op_diff}"
            )

        _configure_grad_checkpointing()
        checkpointing_enabled = True

        model.train()

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        t0 = time.time()

        history, snapshots = (
            _run_optimizer_with_snapshots(
                trainable_named,
                order_seed=(
                    order_seed
                ),
                lr=lr,
                checkpoint_steps=(
                    requested
                ),
            )
        )

        train_wall_seconds = float(
            time.time()
            - t0
        )

        peak_gpu_gib = float(
            torch.cuda.max_memory_allocated()
            / 2**30
        )

        rows.append(
            _base_curve_row(
                method=method,
                family="lora",
                lr=lr,
                eval_df=eval_df,
                base_detail=eval_base_detail,
                base_generation_accuracy=(
                    base_generation_accuracy
                ),
                trainable_params=(
                    trainable_params
                ),
            )
        )

        if (
            return_generation_details
            and base_generation_detail
            is not None
        ):
            details[
                0
            ] = (
                base_generation_detail
                .copy()
            )

        for step in requested:
            step = int(step)

            if step == 0:
                continue

            _load_named_snapshot(
                trainable_named,
                snapshots[
                    step
                ],
            )

            model.eval()

            metrics = (
                evaluate_df_against_base(
                    eval_df,
                    eval_base_detail,
                    batch_size=(
                        EVAL_BATCH_SIZE
                    ),
                )
            )

            generation = (
                evaluate_gsm8k_generation(
                    generation_df
                )
            )

            rows.append({
                "method": str(
                    method
                ),
                "family": "lora",
                "lr": float(
                    lr
                ),
                "updates": int(
                    step
                ),
                "trainable_params": int(
                    trainable_params
                ),
                "target_improvement": float(
                    metrics[
                        "target_improvement"
                    ]
                ),
                "control_improvement": float(
                    metrics[
                        "control_improvement"
                    ]
                ),
                "control_abs_shift": float(
                    metrics[
                        "control_abs_shift"
                    ]
                ),
                "specific_gain": float(
                    metrics[
                        "specific_gain"
                    ]
                ),
                "selective_target_gain": float(
                    metrics[
                        "selective_target_gain"
                    ]
                ),
                "relative_target_improvement": float(
                    metrics[
                        "relative_target_improvement"
                    ]
                ),
                "relative_control_improvement": float(
                    metrics[
                        "relative_control_improvement"
                    ]
                ),
                "generation_accuracy": float(
                    generation[
                        "accuracy"
                    ]
                ),
                "target_nll": float(
                    metrics[
                        "target_nll"
                    ]
                ),
                "control_nll": float(
                    metrics[
                        "control_nll"
                    ]
                ),
                "parameter_delta_l2": float(
                    _snapshot_delta_l2(
                        snapshots[
                            0
                        ],
                        snapshots[
                            step
                        ],
                    )
                ),
                "peak_gpu_gib": (
                    peak_gpu_gib
                ),
                "train_wall_seconds": (
                    train_wall_seconds
                ),
            })

            if return_generation_details:
                details[
                    step
                ] = (
                    generation[
                        "detail"
                    ]
                )

        curve = (
            pd.DataFrame(
                rows
            )
            .sort_values(
                "updates"
            )
            .reset_index(
                drop=True
            )
        )

        return (
            curve,
            details,
            history,
        )

    finally:
        try:
            if (
                hasattr(
                    model,
                    "peft_config"
                )
                and adapter_name
                in getattr(
                    model,
                    "peft_config",
                    {},
                )
            ):
                model.delete_adapter(
                    adapter_name
                )
        finally:
            if checkpointing_enabled:
                _disable_grad_checkpointing()

            for p in model.parameters():
                p.requires_grad_(
                    False
                )

            model.eval()

            gc.collect()
            torch.cuda.empty_cache()

        restored = score_batch(
            probe_batch
        )

        restore_diff = float(
            np.max(
                np.abs(
                    base_probe
                    - restored
                )
            )
        )

        del probe_batch

        if restore_diff > 1e-5:
            raise RuntimeError(
                f"{method}: adapter deletion failed to restore base; "
                f"max ΔNLL={restore_diff}"
            )

## 29 — Baseline autonomous behavior on the two calibration sets

In [ ]:
LR_BASE_GENERATION_PATH = (
    RESULTS / "lr_calibration_base_generation.csv"
)
DOSE_BASE_GENERATION_PATH = (
    RESULTS / "dose_calibration_base_generation.csv"
)

if LR_BASE_GENERATION_PATH.exists():
    LR_BASE_GENERATION_DETAIL = pd.read_csv(
        LR_BASE_GENERATION_PATH
    )
    LR_BASE_GENERATION_ACCURACY = float(
        LR_BASE_GENERATION_DETAIL[
            "correct"
        ].mean()
    )
else:
    out = evaluate_gsm8k_generation(
        lr_generation_df
    )
    LR_BASE_GENERATION_ACCURACY = float(
        out[
            "accuracy"
        ]
    )
    LR_BASE_GENERATION_DETAIL = out[
        "detail"
    ]
    atomic_to_csv(
        LR_BASE_GENERATION_DETAIL,
        LR_BASE_GENERATION_PATH,
        index=False,
    )

if DOSE_BASE_GENERATION_PATH.exists():
    DOSE_BASE_GENERATION_DETAIL = pd.read_csv(
        DOSE_BASE_GENERATION_PATH
    )
    DOSE_BASE_GENERATION_ACCURACY = float(
        DOSE_BASE_GENERATION_DETAIL[
            "correct"
        ].mean()
    )
else:
    out = evaluate_gsm8k_generation(
        dose_generation_df
    )
    DOSE_BASE_GENERATION_ACCURACY = float(
        out[
            "accuracy"
        ]
    )
    DOSE_BASE_GENERATION_DETAIL = out[
        "detail"
    ]
    atomic_to_csv(
        DOSE_BASE_GENERATION_DETAIL,
        DOSE_BASE_GENERATION_PATH,
        index=False,
    )

print(
    "LR-calibration base accuracy:",
    LR_BASE_GENERATION_ACCURACY,
)
print(
    "Dose-calibration base accuracy:",
    DOSE_BASE_GENERATION_ACCURACY,
)

## 30 — Behavior-based family LR calibration at a fixed 8-update probe dose

In [ ]:
LR_CURVES_PATH = RESULTS / "family_lr_calibration.csv"

lr_curves = read_checkpoint_csv(
    LR_CURVES_PATH,
    required_columns=[
        "family_anchor",
        "method",
        "family",
        "lr",
        "updates",
        "generation_accuracy",
        "control_abs_shift",
        "target_improvement",
        "trainable_params",
    ],
    dedupe_keys=[
        "family_anchor",
        "lr",
        "updates",
    ],
)

rows = lr_curves.to_dict(
    orient="records"
)

def _lr_curve_complete(
    df,
    anchor,
    lr,
):
    if df.empty:
        return False

    part = df[
        (df["family_anchor"] == anchor)
        & np.isclose(
            df["lr"].astype(float),
            float(lr),
        )
    ]

    return {
        int(x)
        for x in part[
            "updates"
        ].tolist()
    } >= {
        0,
        int(LR_PROBE_UPDATES),
    }


# Expert-family LR anchor: the v9-specific selector, because it was the
# strongest target-selective expert method in v9. The new contrastive selector
# does not get to choose its own special LR.
for lr in EXPERT_LR_GRID:
    lr = float(lr)
    anchor = "expert_family"

    if _lr_curve_complete(
        lr_curves,
        anchor,
        lr,
    ):
        continue

    print(
        "Expert-family LR probe:",
        lr,
    )

    curve, _, history = (
        run_expert_checkpoint_curve(
            method=(
                "gradient_specific_experts"
            ),
            pairs=(
                expert_sets[
                    "gradient_specific_experts"
                ]
            ),
            order_seed=(
                LR_CALIBRATION_SEED
            ),
            lr=lr,
            checkpoint_steps=[
                0,
                LR_PROBE_UPDATES,
            ],
            eval_df=(
                lr_calibration_df
            ),
            eval_base_detail=(
                LR_CAL_BASE_DETAIL
            ),
            generation_df=(
                lr_generation_df
            ),
            base_generation_accuracy=(
                LR_BASE_GENERATION_ACCURACY
            ),
        )
    )

    curve[
        "family_anchor"
    ] = anchor

    # Replace an incomplete prior attempt for this LR.
    lr_curves = lr_curves[
        ~(
            (
                lr_curves[
                    "family_anchor"
                ]
                == anchor
            )
            & np.isclose(
                lr_curves[
                    "lr"
                ].astype(float),
                lr,
            )
        )
    ]

    lr_curves = pd.concat(
        [
            lr_curves,
            curve,
        ],
        ignore_index=True,
    )

    atomic_to_csv(
        lr_curves,
        LR_CURVES_PATH,
        index=False,
    )

    atomic_to_csv(
        pd.DataFrame(
            history
        ),
        RESULTS
        / (
            "lr_probe_history_expert_"
            + str(lr).replace(".", "p")
            + ".csv"
        ),
        index=False,
    )


# LoRA-family LR anchor: standard broad LoRA. The chosen LR is shared by
# standard, random-layer and contrastive LoRA.
for lr in LORA_LR_GRID:
    lr = float(lr)
    anchor = "lora_family"

    if _lr_curve_complete(
        lr_curves,
        anchor,
        lr,
    ):
        continue

    print(
        "LoRA-family LR probe:",
        lr,
    )

    curve, _, history = (
        run_lora_checkpoint_curve(
            method="standard_lora",
            plan=(
                lora_plans[
                    "standard_lora"
                ]
            ),
            order_seed=(
                LR_CALIBRATION_SEED
            ),
            lr=lr,
            checkpoint_steps=[
                0,
                LR_PROBE_UPDATES,
            ],
            eval_df=(
                lr_calibration_df
            ),
            eval_base_detail=(
                LR_CAL_BASE_DETAIL
            ),
            generation_df=(
                lr_generation_df
            ),
            base_generation_accuracy=(
                LR_BASE_GENERATION_ACCURACY
            ),
        )
    )

    curve[
        "family_anchor"
    ] = anchor

    lr_curves = lr_curves[
        ~(
            (
                lr_curves[
                    "family_anchor"
                ]
                == anchor
            )
            & np.isclose(
                lr_curves[
                    "lr"
                ].astype(float),
                lr,
            )
        )
    ]

    lr_curves = pd.concat(
        [
            lr_curves,
            curve,
        ],
        ignore_index=True,
    )

    atomic_to_csv(
        lr_curves,
        LR_CURVES_PATH,
        index=False,
    )

    atomic_to_csv(
        pd.DataFrame(
            history
        ),
        RESULTS
        / (
            "lr_probe_history_lora_"
            + str(lr).replace(".", "p")
            + ".csv"
        ),
        index=False,
    )


lr_curves = pd.read_csv(
    LR_CURVES_PATH
)

expert_choice = choose_behavior_row(
    lr_curves[
        lr_curves[
            "family_anchor"
        ]
        == "expert_family"
    ],
    allow_zero=False,
)

lora_choice = choose_behavior_row(
    lr_curves[
        lr_curves[
            "family_anchor"
        ]
        == "lora_family"
    ],
    allow_zero=False,
)

CHOSEN_EXPERT_LR = float(
    expert_choice[
        "lr"
    ]
)
CHOSEN_LORA_LR = float(
    lora_choice[
        "lr"
    ]
)

FAMILY_LR_PATH = (
    RESULTS
    / "frozen_family_lrs.json"
)

atomic_write_text(
    FAMILY_LR_PATH,
    json.dumps(
        {
            "expert_lr": (
                CHOSEN_EXPERT_LR
            ),
            "lora_lr": (
                CHOSEN_LORA_LR
            ),
            "probe_updates": (
                LR_PROBE_UPDATES
            ),
            "selection_rule": (
                "generation_accuracy desc, "
                "abs(control_improvement) asc, "
                "target_improvement desc, "
                "lr asc"
            ),
            "expert_anchor": (
                "gradient_specific_experts"
            ),
            "lora_anchor": (
                "standard_lora"
            ),
        },
        indent=2,
    ),
)

display(
    lr_curves[
        lr_curves[
            "updates"
        ].astype(int)
        == int(
            LR_PROBE_UPDATES
        )
    ][
        [
            "family_anchor",
            "method",
            "lr",
            "generation_accuracy",
            "control_abs_shift",
            "target_improvement",
        ]
    ].sort_values(
        [
            "family_anchor",
            "generation_accuracy",
            "control_abs_shift",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
)

print(
    "Frozen expert-family LR:",
    CHOSEN_EXPERT_LR,
)
print(
    "Frozen LoRA-family LR:",
    CHOSEN_LORA_LR,
)

## 31 — Method-specific behavior-aligned early stopping on a disjoint calibration set

In [ ]:
METHOD_SPECS = {
    "gradient_experts": {
        "family": "full_expert",
        "lr": CHOSEN_EXPERT_LR,
    },
    "gradient_specific_experts": {
        "family": "full_expert",
        "lr": CHOSEN_EXPERT_LR,
    },
    "contrastive_experts": {
        "family": "full_expert",
        "lr": CHOSEN_EXPERT_LR,
    },
    "standard_lora": {
        "family": "lora",
        "lr": CHOSEN_LORA_LR,
    },
    "random_layer_lora": {
        "family": "lora",
        "lr": CHOSEN_LORA_LR,
    },
    "contrastive_lora": {
        "family": "lora",
        "lr": CHOSEN_LORA_LR,
    },
}

if list(METHOD_SPECS) != PRIMARY_METHODS:
    raise RuntimeError(
        "Method order/config mismatch."
    )

DOSE_CURVES_PATH = (
    RESULTS
    / "method_dose_calibration.csv"
)

dose_curves = read_checkpoint_csv(
    DOSE_CURVES_PATH,
    required_columns=[
        "method",
        "family",
        "lr",
        "updates",
        "generation_accuracy",
        "control_abs_shift",
        "target_improvement",
        "trainable_params",
    ],
    dedupe_keys=[
        "method",
        "updates",
    ],
)

expected_steps = {
    int(x)
    for x in DOSE_CHECKPOINTS
}

def _dose_curve_complete(
    df,
    method,
):
    if df.empty:
        return False

    part = df[
        df[
            "method"
        ]
        == method
    ]

    return {
        int(x)
        for x in part[
            "updates"
        ].tolist()
    } >= expected_steps


for method in PRIMARY_METHODS:
    if _dose_curve_complete(
        dose_curves,
        method,
    ):
        continue

    spec = METHOD_SPECS[
        method
    ]

    print(
        "\nDose calibration:",
        method,
        "LR",
        spec[
            "lr"
        ],
    )

    if spec[
        "family"
    ] == "full_expert":
        curve, _, history = (
            run_expert_checkpoint_curve(
                method=method,
                pairs=(
                    expert_sets[
                        method
                    ]
                ),
                order_seed=(
                    DOSE_CALIBRATION_SEED
                ),
                lr=(
                    spec[
                        "lr"
                    ]
                ),
                checkpoint_steps=(
                    DOSE_CHECKPOINTS
                ),
                eval_df=(
                    dose_calibration_df
                ),
                eval_base_detail=(
                    DOSE_CAL_BASE_DETAIL
                ),
                generation_df=(
                    dose_generation_df
                ),
                base_generation_accuracy=(
                    DOSE_BASE_GENERATION_ACCURACY
                ),
            )
        )
    else:
        curve, _, history = (
            run_lora_checkpoint_curve(
                method=method,
                plan=(
                    lora_plans[
                        method
                    ]
                ),
                order_seed=(
                    DOSE_CALIBRATION_SEED
                ),
                lr=(
                    spec[
                        "lr"
                    ]
                ),
                checkpoint_steps=(
                    DOSE_CHECKPOINTS
                ),
                eval_df=(
                    dose_calibration_df
                ),
                eval_base_detail=(
                    DOSE_CAL_BASE_DETAIL
                ),
                generation_df=(
                    dose_generation_df
                ),
                base_generation_accuracy=(
                    DOSE_BASE_GENERATION_ACCURACY
                ),
            )
        )

    dose_curves = dose_curves[
        dose_curves[
            "method"
        ]
        != method
    ]

    dose_curves = pd.concat(
        [
            dose_curves,
            curve,
        ],
        ignore_index=True,
    )

    atomic_to_csv(
        dose_curves,
        DOSE_CURVES_PATH,
        index=False,
    )

    atomic_to_csv(
        pd.DataFrame(
            history
        ),
        RESULTS
        / (
            "dose_calibration_history_"
            + method
            + ".csv"
        ),
        index=False,
    )


dose_curves = pd.read_csv(
    DOSE_CURVES_PATH
)

selected_rows = []

for method in PRIMARY_METHODS:
    part = dose_curves[
        dose_curves[
            "method"
        ]
        == method
    ]

    chosen = choose_behavior_row(
        part,
        allow_zero=True,
    )

    selected_rows.append(
        chosen
    )

dose_selection = pd.DataFrame(
    selected_rows
).reset_index(
    drop=True
)

METHOD_SELECTED_UPDATES = {
    str(r.method): int(
        r.updates
    )
    for r in (
        dose_selection
        .itertuples(
            index=False
        )
    )
}

if set(
    METHOD_SELECTED_UPDATES
) != set(
    PRIMARY_METHODS
):
    raise RuntimeError(
        "Dose selection method coverage mismatch."
    )

DOSE_SELECTION_PATH = (
    RESULTS
    / "frozen_method_doses.json"
)

atomic_write_text(
    DOSE_SELECTION_PATH,
    json.dumps(
        {
            "method_updates": (
                METHOD_SELECTED_UPDATES
            ),
            "selection_rule": (
                "generation_accuracy desc, "
                "abs(control_improvement) asc, "
                "target_improvement desc, "
                "updates asc"
            ),
            "calibration_seed": (
                DOSE_CALIBRATION_SEED
            ),
            "candidate_updates": (
                DOSE_CHECKPOINTS
            ),
            "step_zero_is_allowed": True,
        },
        indent=2,
    ),
)

display(
    dose_selection[
        [
            "method",
            "family",
            "lr",
            "updates",
            "generation_accuracy",
            "control_abs_shift",
            "target_improvement",
            "selective_target_gain",
        ]
    ]
)

print(
    "Frozen method-specific doses:",
    METHOD_SELECTED_UPDATES,
)

## 32 — Freeze every adaptive decision before fresh final-test access

In [ ]:
FROZEN_DECISIONS = {
    "protocol_hash": PROTOCOL_HASH,
    "snapshot_sha256": SNAPSHOT_SHA256,
    "v9_exclusion_sha256": V9_EXCLUSION_SHA256,
    "expert_budget_k": EXPERT_BUDGET_K,
    "expert_target_params": EXPERT_TARGET_PARAMS,
    "expert_sets": {
        k: [
            list(
                map(
                    int,
                    p,
                )
            )
            for p in v
        ]
        for k, v in expert_sets.items()
    },
    "lora_plans": lora_plans,
    "chosen_expert_lr": CHOSEN_EXPERT_LR,
    "chosen_lora_lr": CHOSEN_LORA_LR,
    "method_selected_updates": METHOD_SELECTED_UPDATES,
    "final_order_seeds": FINAL_ORDER_SEEDS,
    "final_example_ids": (
        benchmark_df[
            benchmark_df[
                "split"
            ]
            == "final"
        ][
            "example_id"
        ].tolist()
    ),
    "primary_methods": PRIMARY_METHODS,
    "primary_endpoint": (
        "mean fresh-final GSM8K exact final-answer accuracy gain vs base"
    ),
    "secondary_endpoints": [
        "paired-bootstrap accuracy-gain CI over final examples",
        "target teacher-forced NLL improvement",
        "absolute MBPP control NLL shift",
        "selective target gain = target improvement - abs(control improvement)",
        "trainable parameters",
        "selected optimizer updates",
    ],
}

FROZEN_DECISIONS_PATH = (
    RESULTS
    / "FROZEN_BEFORE_FINAL_TEST.json"
)

atomic_write_text(
    FROZEN_DECISIONS_PATH,
    json.dumps(
        FROZEN_DECISIONS,
        indent=2,
    ),
)

if not FROZEN_DECISIONS_PATH.exists():
    raise RuntimeError(
        "Could not freeze final decisions."
    )

print(
    "FINAL TEST LOCKED: selectors, LRs, doses, seeds, and IDs are frozen."
)
print(
    "No final metric has been scored before this point."
)

## 33 — Fresh v9-disjoint base final metrics (first final-test scoring)

In [ ]:
final_test_df = (
    benchmark_df[
        benchmark_df[
            "split"
        ]
        == "final"
    ]
    .reset_index(
        drop=True
    )
)

final_generation_df = (
    final_test_df[
        final_test_df[
            "kind"
        ]
        == "target"
    ]
    .reset_index(
        drop=True
    )
)

if len(final_generation_df) != FINAL_TARGET_N:
    raise RuntimeError(
        "Fresh final target size mismatch."
    )

if (
    int(
        (
            final_test_df[
                "kind"
            ]
            == "control"
        ).sum()
    )
    != FINAL_CONTROL_N
):
    raise RuntimeError(
        "Fresh final control size mismatch."
    )

frozen_ids = set(
    FROZEN_DECISIONS[
        "final_example_ids"
    ]
)

if set(
    final_test_df[
        "example_id"
    ]
) != frozen_ids:
    raise RuntimeError(
        "Final IDs differ from the pre-final freeze file."
    )

FINAL_BASE_DETAIL_PATH = (
    RESULTS
    / "fresh_final_base_detail.npz"
)

BASE_GENERATION_PATH = (
    RESULTS
    / "fresh_final_base_generation.csv"
)

if FINAL_BASE_DETAIL_PATH.exists():
    saved = np.load(
        FINAL_BASE_DETAIL_PATH
    )
    FINAL_BASE_DETAIL = {
        k: saved[
            k
        ]
        for k in [
            "nll",
            "token_acc",
            "seq_exact",
        ]
    }
else:
    batches = build_scoring_batches(
        final_test_df,
        batch_size=EVAL_BATCH_SIZE,
    )
    FINAL_BASE_DETAIL = (
        score_batches_detailed(
            batches
        )
    )
    del batches
    np.savez(
        FINAL_BASE_DETAIL_PATH,
        **FINAL_BASE_DETAIL,
    )

if BASE_GENERATION_PATH.exists():
    BASE_GENERATION_DETAIL = pd.read_csv(
        BASE_GENERATION_PATH
    )
    BASE_GENERATION_ACCURACY = float(
        BASE_GENERATION_DETAIL[
            "correct"
        ].mean()
    )
else:
    base_generation = (
        evaluate_gsm8k_generation(
            final_generation_df
        )
    )
    BASE_GENERATION_ACCURACY = float(
        base_generation[
            "accuracy"
        ]
    )
    BASE_GENERATION_DETAIL = (
        base_generation[
            "detail"
        ]
    )
    atomic_to_csv(
        BASE_GENERATION_DETAIL,
        BASE_GENERATION_PATH,
        index=False,
    )

final_target_mask = (
    final_test_df[
        "kind"
    ].values
    == "target"
)
final_control_mask = (
    final_test_df[
        "kind"
    ].values
    == "control"
)

print(
    "Fresh base GSM8K final NLL:",
    float(
        FINAL_BASE_DETAIL[
            "nll"
        ][
            final_target_mask
        ].mean()
    ),
)
print(
    "Fresh base MBPP control NLL:",
    float(
        FINAL_BASE_DETAIL[
            "nll"
        ][
            final_control_mask
        ].mean()
    ),
)
print(
    "Fresh base GSM8K exact accuracy:",
    BASE_GENERATION_ACCURACY,
    "n=",
    len(
        final_generation_df
    ),
)

## 34 — Fresh final confirmation: 6 methods × 3 seeds at frozen behavior-selected doses

In [ ]:
FINAL_RESULTS_PATH = (
    RESULTS
    / "fresh_final_results.csv"
)

final_results = read_checkpoint_csv(
    FINAL_RESULTS_PATH,
    required_columns=[
        "method",
        "order_seed",
        "family",
        "lr",
        "updates",
        "trainable_params",
        "generation_accuracy",
        "target_improvement",
        "control_improvement",
        "control_abs_shift",
    ],
    dedupe_keys=[
        "method",
        "order_seed",
    ],
)

completed = {
    (
        str(r.method),
        int(r.order_seed),
    )
    for r in final_results.itertuples(
        index=False
    )
}

rows = final_results.to_dict(
    orient="records"
)

for method in PRIMARY_METHODS:
    spec = METHOD_SPECS[
        method
    ]
    selected_updates = int(
        METHOD_SELECTED_UPDATES[
            method
        ]
    )
    lr = float(
        spec[
            "lr"
        ]
    )

    for seed in FINAL_ORDER_SEEDS:
        key = (
            method,
            int(seed),
        )

        if key in completed:
            continue

        print(
            "\nFRESH FINAL",
            method,
            "seed",
            seed,
            "updates",
            selected_updates,
            "lr",
            lr,
        )

        if selected_updates == 0:
            trainable_params = (
                EXPERT_TARGET_PARAMS
                if spec[
                    "family"
                ]
                == "full_expert"
                else int(
                    lora_plans[
                        method
                    ][
                        "actual_trainable_params"
                    ]
                )
            )

            result = _base_curve_row(
                method=method,
                family=(
                    spec[
                        "family"
                    ]
                ),
                lr=lr,
                eval_df=(
                    final_test_df
                ),
                base_detail=(
                    FINAL_BASE_DETAIL
                ),
                base_generation_accuracy=(
                    BASE_GENERATION_ACCURACY
                ),
                trainable_params=(
                    trainable_params
                ),
            )

            result[
                "parameter_delta_l2"
            ] = 0.0
            result[
                "peak_gpu_gib"
            ] = np.nan
            result[
                "train_wall_seconds"
            ] = 0.0

            detail = (
                BASE_GENERATION_DETAIL
                .copy()
            )
            history = []

        elif spec[
            "family"
        ] == "full_expert":
            curve, details, history = (
                run_expert_checkpoint_curve(
                    method=method,
                    pairs=(
                        expert_sets[
                            method
                        ]
                    ),
                    order_seed=(
                        seed
                    ),
                    lr=lr,
                    checkpoint_steps=[
                        0,
                        selected_updates,
                    ],
                    eval_df=(
                        final_test_df
                    ),
                    eval_base_detail=(
                        FINAL_BASE_DETAIL
                    ),
                    generation_df=(
                        final_generation_df
                    ),
                    base_generation_accuracy=(
                        BASE_GENERATION_ACCURACY
                    ),
                    base_generation_detail=(
                        BASE_GENERATION_DETAIL
                    ),
                    return_generation_details=True,
                )
            )

            result = (
                curve[
                    curve[
                        "updates"
                    ].astype(int)
                    == selected_updates
                ]
                .iloc[
                    0
                ]
                .to_dict()
            )

            detail = details[
                selected_updates
            ]

        else:
            curve, details, history = (
                run_lora_checkpoint_curve(
                    method=method,
                    plan=(
                        lora_plans[
                            method
                        ]
                    ),
                    order_seed=(
                        seed
                    ),
                    lr=lr,
                    checkpoint_steps=[
                        0,
                        selected_updates,
                    ],
                    eval_df=(
                        final_test_df
                    ),
                    eval_base_detail=(
                        FINAL_BASE_DETAIL
                    ),
                    generation_df=(
                        final_generation_df
                    ),
                    base_generation_accuracy=(
                        BASE_GENERATION_ACCURACY
                    ),
                    base_generation_detail=(
                        BASE_GENERATION_DETAIL
                    ),
                    return_generation_details=True,
                )
            )

            result = (
                curve[
                    curve[
                        "updates"
                    ].astype(int)
                    == selected_updates
                ]
                .iloc[
                    0
                ]
                .to_dict()
            )

            detail = details[
                selected_updates
            ]

        result[
            "order_seed"
        ] = int(
            seed
        )
        result[
            "selected_updates"
        ] = int(
            selected_updates
        )
        result[
            "base_generation_accuracy"
        ] = float(
            BASE_GENERATION_ACCURACY
        )
        result[
            "generation_accuracy_gain"
        ] = float(
            result[
                "generation_accuracy"
            ]
            - BASE_GENERATION_ACCURACY
        )

        # Per-run artifacts are durable before the completion row is written.
        atomic_to_csv(
            detail,
            RESULTS
            / (
                f"fresh_generation_{method}_seed{int(seed)}.csv"
            ),
            index=False,
        )

        atomic_to_csv(
            pd.DataFrame(
                history
            ),
            RESULTS
            / (
                f"fresh_train_history_{method}_seed{int(seed)}.csv"
            ),
            index=False,
        )

        rows.append(
            result
        )

        final_results = pd.DataFrame(
            rows
        )

        atomic_to_csv(
            final_results,
            FINAL_RESULTS_PATH,
            index=False,
        )

        completed.add(
            key
        )

        print(
            " accuracy:",
            f"{result['generation_accuracy']:.3f}",
            "gain",
            f"{result['generation_accuracy_gain']:+.3f}",
        )
        print(
            " target ΔNLL:",
            f"{result['target_improvement']:+.4f}",
            "control |ΔNLL|:",
            f"{result['control_abs_shift']:.4f}",
        )

        gc.collect()
        torch.cuda.empty_cache()


final_results = pd.read_csv(
    FINAL_RESULTS_PATH
)

expected_runs = (
    len(
        PRIMARY_METHODS
    )
    * len(
        FINAL_ORDER_SEEDS
    )
)

if len(
    final_results
) != expected_runs:
    raise RuntimeError(
        "Fresh final experiment incomplete: "
        f"{len(final_results)} != {expected_runs}"
    )

print(
    "Fresh final experiment: COMPLETE"
)

## 35 — Aggregate fresh-final behavior and paired bootstrap uncertainty

In [ ]:
summary = (
    final_results
    .groupby(
        [
            "method",
            "family",
        ],
        as_index=False,
    )
    .agg(
        n_runs=(
            "order_seed",
            "count",
        ),
        selected_updates=(
            "selected_updates",
            "first",
        ),
        lr=(
            "lr",
            "first",
        ),
        trainable_params=(
            "trainable_params",
            "mean",
        ),
        generation_accuracy_mean=(
            "generation_accuracy",
            "mean",
        ),
        generation_accuracy_std=(
            "generation_accuracy",
            "std",
        ),
        generation_accuracy_gain_mean=(
            "generation_accuracy_gain",
            "mean",
        ),
        target_improvement_mean=(
            "target_improvement",
            "mean",
        ),
        control_abs_shift_mean=(
            "control_abs_shift",
            "mean",
        ),
        selective_target_gain_mean=(
            "selective_target_gain",
            "mean",
        ),
        parameter_delta_l2_mean=(
            "parameter_delta_l2",
            "mean",
        ),
        train_wall_seconds_mean=(
            "train_wall_seconds",
            "mean",
        ),
    )
)


base_detail = (
    BASE_GENERATION_DETAIL[
        [
            "example_id",
            "correct",
        ]
    ]
    .sort_values(
        "example_id"
    )
    .reset_index(
        drop=True
    )
)

base_ids = base_detail[
    "example_id"
].tolist()
base_correct = base_detail[
    "correct"
].to_numpy(
    dtype=np.float64
)


def _method_correct_matrix(
    method,
):
    rows = []

    for seed in FINAL_ORDER_SEEDS:
        path = (
            RESULTS
            / (
                f"fresh_generation_{method}_seed{int(seed)}.csv"
            )
        )

        detail = (
            pd.read_csv(
                path
            )[
                [
                    "example_id",
                    "correct",
                ]
            ]
            .sort_values(
                "example_id"
            )
            .reset_index(
                drop=True
            )
        )

        if detail[
            "example_id"
        ].tolist() != base_ids:
            raise RuntimeError(
                f"{method} seed {seed}: final generation IDs differ from base."
            )

        rows.append(
            detail[
                "correct"
            ].to_numpy(
                dtype=np.float64
            )
        )

    return np.stack(
        rows,
        axis=0,
    )


def _paired_two_way_bootstrap_ci(
    method_matrix,
    base_vector,
    seed,
    n_boot=20000,
):
    """
    Resample BOTH training seeds and final examples.

    This avoids the v10 report pretending that 192 examples are the only source
    of uncertainty when each method also has only three independently trained seeds.
    """
    method_matrix = np.asarray(
        method_matrix,
        dtype=np.float64,
    )
    base_vector = np.asarray(
        base_vector,
        dtype=np.float64,
    )

    if (
        method_matrix.ndim != 2
        or method_matrix.shape[
            1
        ]
        != len(
            base_vector
        )
    ):
        raise RuntimeError(
            "Bootstrap matrix/base shape mismatch."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    n_seed, n_example = (
        method_matrix.shape
    )

    seed_draws = rng.integers(
        0,
        n_seed,
        size=(
            int(n_boot),
            n_seed,
        ),
    )

    example_draws = rng.integers(
        0,
        n_example,
        size=(
            int(n_boot),
            n_example,
        ),
    )

    # Shape: [bootstrap, sampled_seed, example].
    sampled_seed_predictions = (
        method_matrix[
            seed_draws
        ]
    )

    sampled_method_mean = (
        sampled_seed_predictions
        .mean(
            axis=1
        )
    )

    sampled_method_accuracy = (
        np.take_along_axis(
            sampled_method_mean,
            example_draws,
            axis=1,
        )
        .mean(
            axis=1
        )
    )

    sampled_base_accuracy = (
        base_vector[
            example_draws
        ]
        .mean(
            axis=1
        )
    )

    gains = (
        sampled_method_accuracy
        - sampled_base_accuracy
    )

    return (
        float(
            np.quantile(
                gains,
                0.025,
            )
        ),
        float(
            np.quantile(
                gains,
                0.975,
            )
        ),
    )


def _paired_two_way_pairwise_ci(
    matrix_a,
    matrix_b,
    seed,
    n_boot=20000,
):
    matrix_a = np.asarray(
        matrix_a,
        dtype=np.float64,
    )
    matrix_b = np.asarray(
        matrix_b,
        dtype=np.float64,
    )

    if matrix_a.shape != matrix_b.shape:
        raise RuntimeError(
            "Pairwise bootstrap shape mismatch."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    n_seed, n_example = (
        matrix_a.shape
    )

    # Same seed resample for both methods preserves the matched training-order design.
    seed_draws = rng.integers(
        0,
        n_seed,
        size=(
            int(n_boot),
            n_seed,
        ),
    )

    example_draws = rng.integers(
        0,
        n_example,
        size=(
            int(n_boot),
            n_example,
        ),
    )

    mean_a = (
        matrix_a[
            seed_draws
        ]
        .mean(
            axis=1
        )
    )

    mean_b = (
        matrix_b[
            seed_draws
        ]
        .mean(
            axis=1
        )
    )

    acc_a = (
        np.take_along_axis(
            mean_a,
            example_draws,
            axis=1,
        )
        .mean(
            axis=1
        )
    )

    acc_b = (
        np.take_along_axis(
            mean_b,
            example_draws,
            axis=1,
        )
        .mean(
            axis=1
        )
    )

    diff = (
        acc_a
        - acc_b
    )

    return (
        float(
            np.quantile(
                diff,
                0.025,
            )
        ),
        float(
            np.quantile(
                diff,
                0.975,
            )
        ),
    )


bootstrap_rows = []

for method in PRIMARY_METHODS:
    matrix = _method_correct_matrix(
        method
    )

    per_example_delta = (
        matrix.mean(
            axis=0
        )
        - base_correct
    )

    mean_gain = float(
        per_example_delta.mean()
    )

    ci_low, ci_high = (
        _paired_two_way_bootstrap_ci(
            matrix,
            base_correct,
            seed=(
                81000
                + PRIMARY_METHODS.index(
                    method
                )
            ),
        )
    )

    seed_gains = (
        matrix.mean(
            axis=1
        )
        - base_correct.mean()
    )

    bootstrap_rows.append({
        "method": method,
        "paired_mean_accuracy_gain": (
            mean_gain
        ),
        "paired_two_way_ci_low": (
            ci_low
        ),
        "paired_two_way_ci_high": (
            ci_high
        ),
        "positive_seed_count": int(
            np.sum(
                seed_gains
                > 0
            )
        ),
        "nonnegative_seed_count": int(
            np.sum(
                seed_gains
                >= 0
            )
        ),
    })


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

summary = (
    summary.merge(
        bootstrap_df,
        on="method",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "generation_accuracy_mean",
            "control_abs_shift_mean",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)

if not np.allclose(
    summary[
        "generation_accuracy_gain_mean"
    ],
    summary[
        "paired_mean_accuracy_gain"
    ],
    atol=1e-12,
    rtol=0.0,
):
    raise RuntimeError(
        "Aggregate and paired accuracy-gain calculations disagree."
    )

SUMMARY_PATH = (
    RESULTS
    / "fresh_final_summary.csv"
)

atomic_to_csv(
    summary,
    SUMMARY_PATH,
    index=False,
)

display(
    summary
)


def pairwise_bootstrap(
    method_a,
    method_b,
    seed,
):
    matrix_a = _method_correct_matrix(
        method_a
    )
    matrix_b = _method_correct_matrix(
        method_b
    )

    delta = (
        matrix_a.mean(
            axis=0
        )
        - matrix_b.mean(
            axis=0
        )
    )

    low, high = (
        _paired_two_way_pairwise_ci(
            matrix_a,
            matrix_b,
            seed=seed,
        )
    )

    return {
        "method_a": method_a,
        "method_b": method_b,
        "mean_accuracy_difference": float(
            delta.mean()
        ),
        "bootstrap_ci_low": low,
        "bootstrap_ci_high": high,
    }


pairwise = pd.DataFrame([
    pairwise_bootstrap(
        "contrastive_experts",
        "gradient_experts",
        92001,
    ),
    pairwise_bootstrap(
        "contrastive_experts",
        "gradient_specific_experts",
        92002,
    ),
    pairwise_bootstrap(
        "contrastive_lora",
        "standard_lora",
        92003,
    ),
    pairwise_bootstrap(
        "contrastive_lora",
        "random_layer_lora",
        92004,
    ),
])

PAIRWISE_PATH = (
    RESULTS
    / "fresh_final_pairwise_bootstrap.csv"
)

atomic_to_csv(
    pairwise,
    PAIRWISE_PATH,
    index=False,
)

display(
    pairwise
)

# Calibration dose-response plot: this is calibration data, not final.
fig, ax = plt.subplots(
    figsize=(
        12,
        6,
    )
)

for method in PRIMARY_METHODS:
    part = (
        dose_curves[
            dose_curves[
                "method"
            ]
            == method
        ]
        .sort_values(
            "updates"
        )
    )

    ax.plot(
        part[
            "updates"
        ],
        part[
            "generation_accuracy"
        ],
        marker="o",
        label=method,
    )

ax.axhline(
    DOSE_BASE_GENERATION_ACCURACY,
    linestyle="--",
    linewidth=1,
    label="base",
)

ax.set_xlabel(
    "Optimizer updates"
)
ax.set_ylabel(
    "Dose-calibration GSM8K exact accuracy"
)
ax.set_title(
    "Behavioral dose-response curves"
)
ax.legend(
    fontsize=8
)
fig.tight_layout()

fig.savefig(
    RESULTS
    / "dose_calibration_accuracy_curves.png",
    dpi=160,
)

plt.show()

## 36 — Automatic evidence report with no NLL-overclaiming

In [ ]:
def _fmt_ci(
    low,
    high,
):
    return (
        f"[{low:+.4f}, {high:+.4f}]"
    )


report = [
    "# Laguna XS.2 v10 behavior-aligned writeability report",
    "",
    f"Protocol hash: `{PROTOCOL_HASH}`",
    f"Fresh snapshot SHA256: `{SNAPSHOT_SHA256}`",
    f"v9 exclusion SHA256: `{V9_EXCLUSION_SHA256}`",
    "",
    "## Freshness",
    "",
    "- Every v9-used GSM8K/MBPP row was reconstructed from the pinned datasets and excluded.",
    "- v9/v10 example-ID overlap: **0**.",
    f"- Fresh final GSM8K rows: **{FINAL_TARGET_N}**.",
    f"- Fresh final MBPP controls: **{FINAL_CONTROL_N}**.",
    "",
    "## Behavioral calibration",
    "",
    f"- Expert-family LR: `{CHOSEN_EXPERT_LR}`.",
    f"- LoRA-family LR: `{CHOSEN_LORA_LR}`.",
    f"- Candidate doses: `{DOSE_CHECKPOINTS}`.",
    "- LR selection and dose selection used autonomous GSM8K accuracy first.",
    "- `abs(control_improvement)` was the first tie-breaker, so broad control movement is never rewarded.",
    "",
    "## Frozen doses",
    "",
]

for method in PRIMARY_METHODS:
    report.append(
        f"- `{method}`: {METHOD_SELECTED_UPDATES[method]} optimizer updates."
    )

report += [
    "",
    "## Fresh final",
    "",
    f"Base GSM8K exact accuracy: **{BASE_GENERATION_ACCURACY:.4f}**.",
    "",
    summary.to_markdown(
        index=False
    ),
    "",
    "## Pairwise frozen-policy comparisons",
    "",
    pairwise.to_markdown(
        index=False
    ),
    "",
    "## Pre-registered interpretation",
    "",
    (
        "A method is a strong behavioral success only if its fresh-final mean "
        "accuracy gain is positive and its paired two-way bootstrap interval excludes zero."
    ),
    (
        "Teacher-forced target NLL remains secondary: v9 showed that large NLL "
        "movement can coexist with worse autonomous reasoning."
    ),
    (
        "Control movement is reported as an absolute shift because either direction "
        "indicates a less task-specific intervention."
    ),
]

for method in [
    "contrastive_experts",
    "contrastive_lora",
]:
    r = summary[
        summary[
            "method"
        ]
        == method
    ].iloc[
        0
    ]

    success = bool(
        r[
            "paired_two_way_ci_low"
        ]
        > 0
    )

    report += [
        "",
        f"### {method}",
        (
            f"Mean accuracy gain: {r['paired_mean_accuracy_gain']:+.4f}; "
            f"95% paired two-way bootstrap CI "
            f"{_fmt_ci(r['paired_two_way_ci_low'], r['paired_two_way_ci_high'])}."
        ),
        (
            "Strong behavioral success criterion: "
            + (
                "**PASS**."
                if success
                else "**NOT MET**."
            )
        ),
    ]

REPORT_PATH = (
    RESULTS
    / "v10_evidence_report.md"
)

atomic_write_text(
    REPORT_PATH,
    "\n".join(
        report
    ),
)

print(
    "\n".join(
        report[
            :55
        ]
    )
)
print(
    "Full report:",
    REPORT_PATH,
)

## 37 — Manifest, archive, and final validity checklist

In [ ]:
required_files = [
    "benchmark_snapshot.csv",
    "benchmark_snapshot_manifest.json",
    "v9_reconstructed_exclusion_ids.json",
    "protocol_config.json",
    "global_gradient_screen.csv",
    "selection_precise_expert_geometry.csv",
    "frozen_expert_selectors.json",
    "attention_gradient_geometry.csv",
    "lora_budget_plan.json",
    "lr_calibration_base_generation.csv",
    "dose_calibration_base_generation.csv",
    "family_lr_calibration.csv",
    "frozen_family_lrs.json",
    "method_dose_calibration.csv",
    "frozen_method_doses.json",
    "FROZEN_BEFORE_FINAL_TEST.json",
    "fresh_final_base_detail.npz",
    "fresh_final_base_generation.csv",
    "fresh_final_results.csv",
    "fresh_final_summary.csv",
    "fresh_final_pairwise_bootstrap.csv",
    "v10_evidence_report.md",
]

missing = [
    name
    for name in required_files
    if not (
        RESULTS
        / name
    ).exists()
]

if missing:
    raise RuntimeError(
        "Missing required v10 artifacts: "
        + repr(
            missing
        )
    )


# Freshness is a hard validity condition.
all_v9_ids = set().union(
    *v9_used.values()
)

if (
    set(
        benchmark_df[
            "example_id"
        ]
    )
    & all_v9_ids
):
    raise RuntimeError(
        "v9/v10 freshness violation."
    )


# Family-LR calibration completeness.
lr_check = pd.read_csv(
    RESULTS
    / "family_lr_calibration.csv"
)

for anchor, grid in [
    (
        "expert_family",
        EXPERT_LR_GRID,
    ),
    (
        "lora_family",
        LORA_LR_GRID,
    ),
]:
    part = lr_check[
        (
            lr_check[
                "family_anchor"
            ]
            == anchor
        )
        & (
            lr_check[
                "updates"
            ].astype(int)
            == int(
                LR_PROBE_UPDATES
            )
        )
    ]

    observed = sorted(
        part[
            "lr"
        ].astype(float).tolist()
    )

    expected = sorted(
        map(
            float,
            grid,
        )
    )

    if (
        len(observed)
        != len(expected)
        or not np.allclose(
            observed,
            expected,
            rtol=0.0,
            atol=1e-15,
        )
    ):
        raise RuntimeError(
            f"Incomplete LR calibration for {anchor}."
        )


# Dose calibration completeness and exact frozen selections.
dose_check = pd.read_csv(
    RESULTS
    / "method_dose_calibration.csv"
)

for method in PRIMARY_METHODS:
    part = dose_check[
        dose_check[
            "method"
        ]
        == method
    ]

    observed_steps = {
        int(x)
        for x in part[
            "updates"
        ].tolist()
    }

    if observed_steps != set(
        map(
            int,
            DOSE_CHECKPOINTS,
        )
    ):
        raise RuntimeError(
            f"{method}: incomplete dose curve."
        )

    recomputed = choose_behavior_row(
        part,
        allow_zero=True,
    )

    if int(
        recomputed[
            "updates"
        ]
    ) != int(
        METHOD_SELECTED_UPDATES[
            method
        ]
    ):
        raise RuntimeError(
            f"{method}: frozen dose does not match selection rule."
        )


# Final run coverage.
check = pd.read_csv(
    RESULTS
    / "fresh_final_results.csv"
)

if set(
    check[
        "method"
    ]
) != set(
    PRIMARY_METHODS
):
    raise RuntimeError(
        "Fresh final method coverage mismatch."
    )

counts = (
    check.groupby(
        "method"
    )[
        "order_seed"
    ]
    .nunique()
)

if not bool(
    (
        counts
        == len(
            FINAL_ORDER_SEEDS
        )
    ).all()
):
    raise RuntimeError(
        "One or more final methods are missing seeds."
    )

if set(
    check[
        "order_seed"
    ].astype(int)
) != set(
    map(
        int,
        FINAL_ORDER_SEEDS,
    )
):
    raise RuntimeError(
        "Unexpected final seed."
    )

for method in PRIMARY_METHODS:
    part = check[
        check[
            "method"
        ]
        == method
    ]

    expected_updates = int(
        METHOD_SELECTED_UPDATES[
            method
        ]
    )

    if not bool(
        (
            part[
                "selected_updates"
            ].astype(int)
            == expected_updates
        ).all()
    ):
        raise RuntimeError(
            f"{method}: final dose differs from frozen calibration."
        )

    expected_lr = float(
        METHOD_SPECS[
            method
        ][
            "lr"
        ]
    )

    if not bool(
        np.isclose(
            part[
                "lr"
            ].astype(float),
            expected_lr,
            rtol=0.0,
            atol=1e-15,
        ).all()
    ):
        raise RuntimeError(
            f"{method}: final LR differs from frozen family LR."
        )


# Parameter-budget validity.
if check[
    "trainable_params"
].le(
    0
).any():
    raise RuntimeError(
        "A final arm has zero nominal trainable parameters."
    )

budget_error = abs(
    check[
        "trainable_params"
    ]
    - EXPERT_TARGET_PARAMS
) / float(
    EXPERT_TARGET_PARAMS
)

if (
    budget_error
    > MAX_LORA_BUDGET_REL_ERROR
).any():
    raise RuntimeError(
        "A final arm violates the matched parameter budget."
    )


# Symmetric control metric is exact.
if not np.allclose(
    check[
        "control_abs_shift"
    ].to_numpy(
        dtype=np.float64
    ),
    np.abs(
        check[
            "control_improvement"
        ].to_numpy(
            dtype=np.float64
        )
    ),
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "control_abs_shift is inconsistent."
    )


numeric_cols = [
    "generation_accuracy",
    "generation_accuracy_gain",
    "target_improvement",
    "control_improvement",
    "control_abs_shift",
    "selective_target_gain",
]

if not bool(
    np.isfinite(
        check[
            numeric_cols
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
):
    raise RuntimeError(
        "Non-finite final metric."
    )


# Every final generation artifact must contain exactly the fresh target IDs.
expected_generation_ids = set(
    final_generation_df[
        "example_id"
    ]
)

for r in check.itertuples(
    index=False
):
    generation_path = (
        RESULTS
        / (
            f"fresh_generation_{r.method}_seed{int(r.order_seed)}.csv"
        )
    )

    history_path = (
        RESULTS
        / (
            f"fresh_train_history_{r.method}_seed{int(r.order_seed)}.csv"
        )
    )

    if not generation_path.exists():
        raise RuntimeError(
            f"Missing {generation_path.name}"
        )

    if not history_path.exists():
        raise RuntimeError(
            f"Missing {history_path.name}"
        )

    detail = pd.read_csv(
        generation_path
    )

    if (
        len(
            detail
        )
        != FINAL_TARGET_N
        or set(
            detail[
                "example_id"
            ]
        )
        != expected_generation_ids
    ):
        raise RuntimeError(
            f"{generation_path.name}: final ID coverage mismatch."
        )


summary_check = pd.read_csv(
    RESULTS
    / "fresh_final_summary.csv"
)

if len(
    summary_check
) != len(
    PRIMARY_METHODS
):
    raise RuntimeError(
        "Final summary row count mismatch."
    )

if not bool(
    np.isfinite(
        summary_check[
            [
                "paired_mean_accuracy_gain",
                "paired_two_way_ci_low",
                "paired_two_way_ci_high",
            ]
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
):
    raise RuntimeError(
        "Non-finite bootstrap summary."
    )


manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "protocol_version": PROTOCOL_VERSION,
    "protocol_hash": PROTOCOL_HASH,
    "snapshot_sha256": SNAPSHOT_SHA256,
    "v9_exclusion_sha256": V9_EXCLUSION_SHA256,
    "v9_v10_overlap_count": 0,
    "model_id": MODEL_ID,
    "transformers": package_version(
        "transformers"
    ),
    "peft": package_version(
        "peft"
    ),
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(
        0
    ),
    "expert_target_params": (
        EXPERT_TARGET_PARAMS
    ),
    "chosen_expert_lr": (
        CHOSEN_EXPERT_LR
    ),
    "chosen_lora_lr": (
        CHOSEN_LORA_LR
    ),
    "method_selected_updates": (
        METHOD_SELECTED_UPDATES
    ),
    "methods": PRIMARY_METHODS,
    "final_seeds": FINAL_ORDER_SEEDS,
    "fresh_final_target_n": FINAL_TARGET_N,
    "fresh_final_control_n": FINAL_CONTROL_N,
    "files": {
        name: sha256_file(
            RESULTS
            / name
        )
        for name in required_files
    },
}

atomic_write_text(
    RESULTS
    / "manifest.json",
    json.dumps(
        manifest,
        indent=2,
    ),
)

archive_path = (
    RESULTS
    / "laguna_xs2_v10_behavior_aligned_results.zip"
)

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=(
        zipfile.ZIP_DEFLATED
    ),
) as archive:
    for path in sorted(
        RESULTS.iterdir()
    ):
        if (
            path.is_file()
            and path
            != archive_path
        ):
            archive.write(
                path,
                arcname=(
                    path.name
                ),
            )

print(
    "V10 VALIDITY CHECKLIST: PASS"
)
print(
    "v9/v10 overlap: 0"
)
print(
    "Results archive:",
    archive_path,
)